# KBO V18 — Temporal-Stable Conditional Residual · Recovered Exact Pipeline

Recovered from the embedded V18 champion baseline and V18 deployment stage inside the supplied V20 notebook. All executable code cells are preserved unchanged from that notebook; V20-only challenger stages are omitted.


In [ ]:
from __future__ import annotations
import os, gc, json, math, time, random, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',260)
pd.set_option('display.width',300)

PROTOCOL_VERSION='V20_TEMPORAL_ROLE_USAGE_V1'
TARGET='control_success'
ANCHOR_DISCOVERY_MAX_SEASON=2022
BASE_SOURCE_YEAR=2023
VALID_YEAR=2024
CURRENT_LB=1081.6824647538
TARGET_LB=1200.0
SMOKE_MODE=os.environ.get('V20_SMOKE','0')=='1'
N_JOBS=max(1,min(6,os.cpu_count() or 1))
REPORT_DIR=Path('/kaggle/working/v20_temporal_role_usage') if Path('/kaggle/working').exists() else Path('./v20_temporal_role_usage')
REPORT_DIR.mkdir(parents=True,exist_ok=True)

BASE_CFG={
 'exact_lgb':dict(n_estimators=120 if not SMOKE_MODE else 20,learning_rate=.035,num_leaves=7,min_child_samples=350 if not SMOKE_MODE else 40,subsample=.9,colsample_bytree=.9,reg_alpha=1.,reg_lambda=6.,max_bin=127,random_state=918),
 'ridge_alpha':10000.0,
 'trend_lgb':dict(n_estimators=180 if not SMOKE_MODE else 25,learning_rate=.04,num_leaves=7,min_child_samples=300 if not SMOKE_MODE else 40,subsample=.85,colsample_bytree=.9,reg_alpha=1.,reg_lambda=5.,max_bin=127,random_state=77),
}
V15_CF_CFG=dict(n_estimators=100 if not SMOKE_MODE else 12,max_depth=10,min_samples_leaf=200 if not SMOKE_MODE else 25,max_features=.7)
FINAL_ADAPTER_CFG=dict(n_estimators=120 if not SMOKE_MODE else 15,max_depth=10,min_samples_leaf=200 if not SMOKE_MODE else 25,max_features=.7)
ADAPTER_SHRINK=0.50
CF_SEEDS=(42,2026,314)

# Protected V18 champion.
V18_HISTORY_YEARS=3
V18_DECAY=0.65
V18_PARENT_STRENGTH=100.0
V18_HAND_STRENGTH=35.0
V18_PRESSURE_HAND_STRENGTH=45.0
V18_RIDGE_ALPHA=1000.0
V18_GAMMA=0.75
V18_GAMMA_STRESS=(0.55,0.65,0.75,0.85,0.95)
V18_ROUTE_DOMAIN='R_CORE'

# V20 role/usage transition specialist. No 2024 value participates in candidate selection.
V20_HISTORY_YEARS=3
V20_DECAY=0.62
V20_VOLUME_K=500.0
V20_ROLE_K=350.0
V20_ALPHA_GRID=(300.0,1000.0,3000.0,10000.0)
V20_GAMMA_GRID=(0.10,0.20,0.35,0.50,0.70)
V20_ROUTE_DOMAIN='R_CORE'
V20_DISCOVERY_TARGETS=(2022,2023)
V20_STRESS_TARGET=2024
V20_MIN_DISCOVERY_GAIN=2.0
V20_MIN_STRESS_GAIN=3.0
V20_MIN_CORE_STRESS_GAIN=4.0
V20_MIN_BOOT_P05=0.5
V20_MIN_BOOT_PROB=.95
V20_MIN_LATE_GAIN=-1.0
V20_WORST_MONTH_GAIN=-10.0

# V18 gate retained exactly as protected baseline requirement.
MIN_GAIN_2023=8.0
MIN_GAIN_2024=10.0
MIN_CORE_GAIN_2024=15.0
MIN_BOOT_P05=3.0
MIN_BOOT_PROB=.97
MIN_LATE_GAIN=0.0

CONFIG_HASH=hashlib.sha1(json.dumps({
 'base':BASE_CFG,'cf':V15_CF_CFG,'final':FINAL_ADAPTER_CFG,'shrink':ADAPTER_SHRINK,
 'stable':dict(history=V18_HISTORY_YEARS,decay=V18_DECAY,parent=V18_PARENT_STRENGTH,hand=V18_HAND_STRENGTH,pb=V18_PRESSURE_HAND_STRENGTH,alpha=V18_RIDGE_ALPHA,gamma=V18_GAMMA,route=V18_ROUTE_DOMAIN),
 'v20':dict(history=V20_HISTORY_YEARS,decay=V20_DECAY,volume_k=V20_VOLUME_K,role_k=V20_ROLE_K,alphas=V20_ALPHA_GRID,gammas=V20_GAMMA_GRID,route=V20_ROUTE_DOMAIN)
},sort_keys=True).encode()).hexdigest()[:16]
print('PROTOCOL_VERSION:',PROTOCOL_VERSION)
print('CONFIG_HASH:',CONFIG_HASH)
print('SMOKE_MODE:',SMOKE_MODE,'CPU jobs:',N_JOBS)
print('CURRENT LB:',CURRENT_LB,'TARGET LB:',TARGET_LB,'GAP:',TARGET_LB-CURRENT_LB)


In [ ]:
# Row-wise features: only official current-row inputs. No target aggregation and no other prediction/test row is used.
def _n(frame,c,default=np.nan):
    if c in frame:return pd.to_numeric(frame[c],errors='coerce').to_numpy(np.float64)
    return np.full(len(frame),default,np.float64)

def build_self_state(frame):
    n=len(frame);z=pd.DataFrame(index=np.arange(n))
    pn=np.maximum(np.nan_to_num(_n(frame,'asof_pitcher_n',0),nan=0.0),0);bn=np.maximum(np.nan_to_num(_n(frame,'asof_batter_n',0),nan=0.0),0)
    career=_n(frame,'asof_pitcher_success_rate',.5);career=np.where(np.isfinite(career),career,.5)
    p1=_n(frame,'asof_pitcher_prev1_game_success_rate');p3=_n(frame,'asof_pitcher_prev3_game_success_rate');p5=_n(frame,'asof_pitcher_prev5_game_success_rate')
    p1=np.where(np.isfinite(p1),p1,career);p3=np.where(np.isfinite(p3),p3,p1);p5=np.where(np.isfinite(p5),p5,p3)
    bsucc=_n(frame,'asof_batter_success_rate',.5);bsucc=np.where(np.isfinite(bsucc),bsucc,.5)
    middle=_n(frame,'asof_pitcher_middle_rate',0);reverse=_n(frame,'asof_pitcher_reverse_rate',0);ball=_n(frame,'asof_pitcher_ball_rate',0);strike=_n(frame,'asof_pitcher_strike_rate',0)
    fast=_n(frame,'asof_pitcher_fastball_rate',1/3);brk=_n(frame,'asof_pitcher_breaking_rate',1/3);off=_n(frame,'asof_pitcher_offspeed_rate',1/3)
    fast=np.where(np.isfinite(fast),fast,1/3);brk=np.where(np.isfinite(brk),brk,1/3);off=np.where(np.isfinite(off),off,1/3);sm=np.maximum(fast+brk+off,1e-6);fast/=sm;brk/=sm;off/=sm
    balls=np.nan_to_num(_n(frame,'balls_before',0));strikes=np.nan_to_num(_n(frame,'strikes_before',0));inning=np.nan_to_num(_n(frame,'inning',1));score=np.nan_to_num(_n(frame,'score_diff_pitcher_team',0));li=np.maximum(np.nan_to_num(_n(frame,'li',0)),0)
    top=frame['top_bottom'].astype(str).to_numpy();hwe=np.nan_to_num(_n(frame,'home_win_expectancy',50),nan=50)/100.;awe=np.nan_to_num(_n(frame,'away_win_expectancy',50),nan=50)/100.;pwe=np.where(top=='T',hwe,awe)
    ph=frame['pitcher_hand'].astype(str).to_numpy();bh=frame['batter_hand'].astype(str).to_numpy()
    z['self__log_pitcher_n']=np.log1p(pn).astype(np.float32);z['self__pitcher_reliability']=(pn/(pn+120)).astype(np.float32)
    z['self__career_success']=career.astype(np.float32);z['self__prev1_success']=p1.astype(np.float32);z['self__prev3_success']=p3.astype(np.float32);z['self__prev5_success']=p5.astype(np.float32)
    z['self__recent_blend']=(.50*p1+.30*p3+.20*p5).astype(np.float32);z['self__prev1_minus_prev5']=(p1-p5).astype(np.float32);z['self__recent_minus_career']=(.5*p1+.3*p3+.2*p5-career).astype(np.float32)
    middle_safe=np.nan_to_num(middle,nan=0.0);m1=_n(frame,'asof_pitcher_prev1_game_middle_rate');m3=_n(frame,'asof_pitcher_prev3_game_middle_rate');m5=_n(frame,'asof_pitcher_prev5_game_middle_rate');m1=np.where(np.isfinite(m1),m1,middle_safe);m3=np.where(np.isfinite(m3),m3,m1);m5=np.where(np.isfinite(m5),m5,m3);z['self__recent_middle_blend']=(.50*m1+.30*m3+.20*m5).astype(np.float32);z['self__recent_middle_delta']=(m1-m5).astype(np.float32)
    z['self__log_batter_n']=np.log1p(bn).astype(np.float32);z['self__batter_reliability']=(bn/(bn+120)).astype(np.float32);z['self__batter_success']=bsucc.astype(np.float32);z['self__pitcher_minus_batter']=(career-bsucc).astype(np.float32)
    z['self__middle']=np.nan_to_num(middle,nan=0).astype(np.float32);z['self__reverse']=np.nan_to_num(reverse,nan=0).astype(np.float32);z['self__strike_ball_margin']=np.nan_to_num(strike-ball,nan=0).astype(np.float32)
    entropy=-(fast*np.log(np.clip(fast,1e-6,1))+brk*np.log(np.clip(brk,1e-6,1))+off*np.log(np.clip(off,1e-6,1)))
    z['self__pitchmix_entropy']=entropy.astype(np.float32);z['self__offspeed']=off.astype(np.float32);z['self__fast_off_logratio']=np.log(np.clip(fast,1e-6,1)/np.clip(off,1e-6,1)).astype(np.float32);z['self__break_off_logratio']=np.log(np.clip(brk,1e-6,1)/np.clip(off,1e-6,1)).astype(np.float32)
    z['self__count_pressure']=(.7*(balls==3)+1.1*((balls==3)&(strikes==2))+.25*(strikes==2)).astype(np.float32);z['self__full_count']=((balls==3)&(strikes==2)).astype(np.float32)
    z['self__inning']=inning.astype(np.float32);z['self__late']=(inning>=7).astype(np.float32);z['self__score_abs']=np.abs(score).astype(np.float32);z['self__li_log1p']=np.log1p(li).astype(np.float32)
    z['self__pitcher_we']=np.clip(pwe,0,1).astype(np.float32);z['self__win_uncertainty']=(1-2*np.abs(np.clip(pwe,0,1)-.5)).astype(np.float32);z['self__same_hand']=(ph==bh).astype(np.float32)
    pressure=z['self__count_pressure'].to_numpy(np.float64);z['self__career_x_pressure']=(career*pressure).astype(np.float32);z['self__recent_x_pressure']=((.5*p1+.3*p3+.2*p5)*pressure).astype(np.float32);z['self__offspeed_x_same_hand']=(off*(ph==bh)).astype(np.float32)
    return z

# ------------------------------------------------------------------
# Global-ASOF-aligned current-season decomposition.
# Official asof_n is global across R/F, so historical subtraction MUST
# use all official history, not one routed domain.
# ------------------------------------------------------------------
P_COMPONENT_RATES=['asof_pitcher_reverse_rate','asof_pitcher_middle_rate','asof_pitcher_ball_rate','asof_pitcher_strike_rate','asof_pitcher_fastball_rate','asof_pitcher_breaking_rate','asof_pitcher_offspeed_rate']
B_COMPONENT_RATES=['asof_batter_middle_rate']

def _rounded_cumulative_count(n,rate):
    n=np.asarray(n,np.float64);rate=np.asarray(rate,np.float64);out=np.zeros(len(n),np.float64);ok=(n>0)&np.isfinite(rate);out[ok]=np.rint(n[ok]*rate[ok]);return out

def build_global_season_bank(history,cutoff):
    h=history.loc[history.season.astype(int)<int(cutoff)].copy()
    prior=float(h.loc[h.season.eq(h.season.max()),TARGET].mean()) if len(h) else .5
    pg=h.groupby('pitcher_id',sort=False)[TARGET].agg(['size','sum']) if len(h) else pd.DataFrame(columns=['size','sum'])
    bg=h.groupby('batter_id',sort=False)[TARGET].agg(['size','sum']) if len(h) else pd.DataFrame(columns=['size','sum'])
    bank={'cutoff':int(cutoff),'prior':prior,'pitcher_n':pg['size'].to_dict(),'pitcher_s':pg['sum'].to_dict(),'batter_n':bg['size'].to_dict(),'batter_s':bg['sum'].to_dict(),'source_max_season':int(h.season.max()) if len(h) else None}

    # Component-rate lineage: asof_n * rate is a rounded cumulative count. The last
    # historical row stores counts BEFORE that last pitch. We therefore use that
    # pre-last boundary as the baseline. The resulting window contains exactly one
    # boundary pitch + current-season pitches, avoiding any need to inspect other test rows.
    prev=h.loc[h.season.eq(h.season.max())] if len(h) else h
    bank['component_priors']={c:float(pd.to_numeric(prev[c],errors='coerce').mean()) if c in prev and pd.to_numeric(prev[c],errors='coerce').notna().any() else 0.5 for c in P_COMPONENT_RATES+B_COMPONENT_RATES}
    if len(h):
        pidx=h.groupby('pitcher_id',sort=False)['asof_pitcher_n'].idxmax();plast=h.loc[pidx,['pitcher_id','asof_pitcher_n']+P_COMPONENT_RATES].set_index('pitcher_id')
        pbase_n=pd.to_numeric(plast['asof_pitcher_n'],errors='coerce').fillna(0).astype(np.float64);bank['pitcher_component_base_n']=pbase_n.to_dict();bank['pitcher_component_base']={}
        for c in P_COMPONENT_RATES:
            rr=pd.to_numeric(plast[c],errors='coerce');cnt=np.rint(pbase_n*rr);bank['pitcher_component_base'][c]=cnt.to_dict()
        bidx=h.groupby('batter_id',sort=False)['asof_batter_n'].idxmax();blast=h.loc[bidx,['batter_id','asof_batter_n']+B_COMPONENT_RATES].set_index('batter_id')
        bbase_n=pd.to_numeric(blast['asof_batter_n'],errors='coerce').fillna(0).astype(np.float64);bank['batter_component_base_n']=bbase_n.to_dict();bank['batter_component_base']={}
        for c in B_COMPONENT_RATES:
            rr=pd.to_numeric(blast[c],errors='coerce');cnt=np.rint(bbase_n*rr);bank['batter_component_base'][c]=cnt.to_dict()
    else:
        bank['pitcher_component_base_n']={};bank['pitcher_component_base']={c:{} for c in P_COMPONENT_RATES};bank['batter_component_base_n']={};bank['batter_component_base']={c:{} for c in B_COMPONENT_RATES}
    return bank

def transform_global_season(bank,rows):
    r=rows.reset_index(drop=True);z=pd.DataFrame(index=np.arange(len(r)));prior=float(bank['prior'])
    def one(ent,ncol,rcol,prefix,kn=80.0):
        ids=r[ent];hn=ids.map(bank[prefix+'_n']).fillna(0).to_numpy(np.float64);hs=ids.map(bank[prefix+'_s']).fillna(0).to_numpy(np.float64)
        an=np.maximum(np.nan_to_num(_n(r,ncol,0),nan=0),0);ar=_n(r,rcol,prior);ar=np.where(np.isfinite(ar),ar,prior)
        # Exact integer reconstruction: official rates are rounded, but n*rate is always
        # within <0.01 of the underlying integer count in the released train.
        cumulative_success=_rounded_cumulative_count(an,ar)
        sn=np.maximum(an-hn,0.0);ss=np.clip(cumulative_success-hs,0.0,sn)
        raw=np.divide(ss,sn,out=np.full(len(r),prior,np.float64),where=sn>0)
        for k in [40.,80.,160.]:z[f'gseason__{prefix}_rate_k{int(k)}']=((ss+k*prior)/(sn+k)).astype(np.float32)
        z[f'gseason__{prefix}_raw']=raw.astype(np.float32);z[f'gseason__{prefix}_log_n']=np.log1p(sn).astype(np.float32);z[f'gseason__{prefix}_reliability']=(sn/(sn+kn)).astype(np.float32);z[f'gseason__{prefix}_minus_career']=(raw-ar).astype(np.float32)
    one('pitcher_id','asof_pitcher_n','asof_pitcher_success_rate','pitcher',80.)
    one('batter_id','asof_batter_n','asof_batter_success_rate','batter',100.)
    z['gseason__pitcher_minus_batter_k80']=(z['gseason__pitcher_rate_k80']-z['gseason__batter_rate_k80']).astype(np.float32)

    # Exact cumulative component counts with one-pitch boundary bridge.
    ids=r['pitcher_id'];an=np.maximum(np.nan_to_num(_n(r,'asof_pitcher_n',0),nan=0),0);base_n=ids.map(bank['pitcher_component_base_n']).fillna(0).to_numpy(np.float64);win_n=np.maximum(an-base_n,0.0)
    comp={}
    for c in P_COMPONENT_RATES:
        pr=float(bank['component_priors'].get(c,0.5));rr=_n(r,c,pr);rr=np.where(np.isfinite(rr),rr,pr);cum=_rounded_cumulative_count(an,rr);base=ids.map(bank['pitcher_component_base'].get(c,{})).fillna(0).to_numpy(np.float64);wc=np.clip(cum-base,0.0,win_n);comp[c]=((wc+60.0*pr)/(win_n+60.0)).astype(np.float32)
    z['gseason__pitcher_component_log_n']=np.log1p(win_n).astype(np.float32);z['gseason__pitcher_component_reliability']=(win_n/(win_n+60.)).astype(np.float32)
    z['gseason__pitcher_strike_rate']=comp['asof_pitcher_strike_rate'];z['gseason__pitcher_ball_rate']=comp['asof_pitcher_ball_rate'];z['gseason__pitcher_strike_ball_margin']=(comp['asof_pitcher_strike_rate']-comp['asof_pitcher_ball_rate']).astype(np.float32)
    z['gseason__pitcher_middle_rate']=comp['asof_pitcher_middle_rate'];z['gseason__pitcher_reverse_rate']=comp['asof_pitcher_reverse_rate']
    fast=np.clip(comp['asof_pitcher_fastball_rate'],1e-6,1);brk=np.clip(comp['asof_pitcher_breaking_rate'],1e-6,1);off=np.clip(comp['asof_pitcher_offspeed_rate'],1e-6,1);sm=np.maximum(fast+brk+off,1e-6);fast=fast/sm;brk=brk/sm;off=off/sm
    z['gseason__pitcher_fastball_rate']=fast.astype(np.float32);z['gseason__pitcher_breaking_rate']=brk.astype(np.float32);z['gseason__pitcher_offspeed_rate']=off.astype(np.float32);z['gseason__pitcher_pitchmix_entropy']=(-(fast*np.log(fast)+brk*np.log(brk)+off*np.log(off))).astype(np.float32)

    bids=r['batter_id'];ban=np.maximum(np.nan_to_num(_n(r,'asof_batter_n',0),nan=0),0);bbase_n=bids.map(bank['batter_component_base_n']).fillna(0).to_numpy(np.float64);bwin=np.maximum(ban-bbase_n,0.0);c='asof_batter_middle_rate';pr=float(bank['component_priors'].get(c,0.5));rr=_n(r,c,pr);rr=np.where(np.isfinite(rr),rr,pr);cum=_rounded_cumulative_count(ban,rr);base=bids.map(bank['batter_component_base'].get(c,{})).fillna(0).to_numpy(np.float64);wc=np.clip(cum-base,0.0,bwin);bmid=((wc+80.*pr)/(bwin+80.)).astype(np.float32)
    z['gseason__batter_middle_rate']=bmid;z['gseason__batter_middle_minus_pitcher_middle']=(bmid-z['gseason__pitcher_middle_rate']).astype(np.float32)
    return z

def global_season_features_for_training(frame):
    pieces=[]
    for year in sorted(frame.season.astype(int).unique()):
        idx=frame.index[frame.season.astype(int).eq(year)];bank=build_global_season_bank(frame.loc[frame.season.astype(int)<year],year);part=transform_global_season(bank,frame.loc[idx]);part.index=idx;pieces.append(part)
    return pd.concat(pieces).sort_index().reindex(frame.index).reset_index(drop=True)


print('Exact-ASOF row-state functions loaded')


In [ ]:
def discover_csv(required,name='train.csv'):
    roots=[Path('/kaggle/input'),Path('/content'),Path('.'),Path('/mnt/data')]
    cand=[]
    for root in roots:
        if not root.exists(): continue
        cand+=list(root.rglob(name))
        if name=='train.csv':cand+=list(root.rglob('train(1).csv'))
    good=[]
    for p in cand:
        try:
            cols=pd.read_csv(p,nrows=2,encoding='utf-8-sig').columns
            if set(required).issubset(cols):good.append(p)
        except Exception:pass
    return sorted(set(good),key=lambda p:p.stat().st_size,reverse=True)[0] if good else None

TRAIN_PATH=discover_csv({'row_id','season',TARGET,'pitcher_id','batter_id','game_type','pitcher_team_id','batter_team_id'},'train.csv')
if TRAIN_PATH is None:raise FileNotFoundError('official train.csv not found')
print('TRAIN_PATH:',TRAIN_PATH)
screen=pd.read_csv(TRAIN_PATH,encoding='utf-8-sig',low_memory=False)
screen['season']=pd.to_numeric(screen['season'],errors='raise').astype(int)
screen['game_type']=screen['game_type'].astype('string').fillna('<NA>').astype(str)
assert screen.row_id.is_unique and TARGET in screen
assert set(screen.game_type.unique())=={'R','F'}

# Router identity is learned ONLY from pre-2023 F structure.
f_hist=screen.loc[screen.game_type.eq('F') & screen.season.le(ANCHOR_DISCOVERY_MAX_SEASON)]
teams=sorted(set(f_hist.pitcher_team_id.unique())|set(f_hist.batter_team_id.unique()))
involvement={int(t):float(((f_hist.pitcher_team_id==t)|(f_hist.batter_team_id==t)).mean()) for t in teams}
anchors=[t for t,v in involvement.items() if v>.999999]
assert len(anchors)==1,('expected one persistent F anchor',involvement)
ANCHOR_TEAM_ID=int(anchors[0])
screen['domain3']=np.where(screen.game_type.eq('F'),'F',np.where((screen.pitcher_team_id==ANCHOR_TEAM_ID)|(screen.batter_team_id==ANCHOR_TEAM_ID),'R_ANCHOR','R_CORE'))
print('ANCHOR_TEAM_ID:',ANCHOR_TEAM_ID)

stats=screen.groupby(['season','domain3'])[TARGET].agg(['size','mean']).reset_index()
print('\n===== YEAR × DOMAIN AUDIT =====');display(stats)
share24=screen.loc[screen.season.eq(VALID_YEAR),'domain3'].value_counts(normalize=True).rename('share').reset_index();display(share24)
print('THREE_DOMAIN_ROUTER_PASS')
print('TRAIN_ROWS:',len(screen),'SEASONS:',sorted(screen.season.unique().tolist()))


In [ ]:
def raw_bss(y,p):
    y=np.asarray(y,dtype=np.float64);p=np.clip(np.asarray(p,dtype=np.float64),0,1)
    ref=max(float(y.mean()*(1-y.mean())),1e-12)
    return float(100000.0*(1.0-np.mean((p-y)**2)/ref))

def pitcher_cluster_bootstrap_gain(frame,y,p_new,p_base,draws=1000,seed=979):
    pid=frame.pitcher_id.astype(str).to_numpy();y=np.asarray(y,float);p_new=np.asarray(p_new,float);p_base=np.asarray(p_base,float)
    codes,uniq=pd.factorize(pid,sort=False);G=len(uniq);n=np.bincount(codes,minlength=G).astype(float);sy=np.bincount(codes,weights=y,minlength=G);en=np.bincount(codes,weights=(p_new-y)**2,minlength=G);eb=np.bincount(codes,weights=(p_base-y)**2,minlength=G)
    rng=np.random.default_rng(seed);g=[]
    for _ in range(int(draws)):
        draw=rng.integers(0,G,size=G);w=np.bincount(draw,minlength=G).astype(float);N=float(w@n)
        if N<100:continue
        ym=float(w@sy/N);ref=max(ym*(1-ym),1e-12);g.append(float(100000.0*((w@eb-w@en)/N)/ref))
    a=np.asarray(g,float)
    return {'mean':float(a.mean()),'median':float(np.median(a)),'p05':float(np.quantile(a,.05)),'p10':float(np.quantile(a,.10)),'p95':float(np.quantile(a,.95)),'prob_positive':float((a>0).mean())}

RAWNUM=['game_month','inning','balls_before','strikes_before','outs_before','score_diff_pitcher_team','num_runners_on','home_win_expectancy','li','asof_pitcher_n','asof_pitcher_success_rate','asof_pitcher_reverse_rate','asof_pitcher_middle_rate','asof_pitcher_ball_rate','asof_pitcher_strike_rate','asof_pitcher_prev1_game_success_rate','asof_pitcher_prev3_game_success_rate','asof_pitcher_prev5_game_success_rate','asof_pitcher_prev1_game_middle_rate','asof_pitcher_prev3_game_middle_rate','asof_pitcher_prev5_game_middle_rate','asof_batter_n','asof_batter_success_rate','asof_batter_middle_rate','asof_pitcher_fastball_rate','asof_pitcher_breaking_rate','asof_pitcher_offspeed_rate']
RAWNUM=[c for c in RAWNUM if c in screen]
CAT_SMALL=[c for c in ['game_dayofweek','top_bottom','game_type','domain3','base_state','pitcher_hand','batter_hand','pitcher_team_id','batter_team_id'] if c in screen]
BANK_COLS=list(dict.fromkeys(['season','pitcher_id','batter_id',TARGET,'asof_pitcher_n','asof_batter_n']+P_COMPONENT_RATES+B_COMPONENT_RATES))

def add_eng(frame):
    x=frame.copy()
    x['eng_recent35']=(.5*pd.to_numeric(x.asof_pitcher_prev1_game_success_rate,errors='coerce')+.3*pd.to_numeric(x.asof_pitcher_prev3_game_success_rate,errors='coerce')+.2*pd.to_numeric(x.asof_pitcher_prev5_game_success_rate,errors='coerce'))
    x['eng_recent_delta']=x.eng_recent35-pd.to_numeric(x.asof_pitcher_success_rate,errors='coerce')
    x['eng_sb']=pd.to_numeric(x.asof_pitcher_strike_rate,errors='coerce')-pd.to_numeric(x.asof_pitcher_ball_rate,errors='coerce')
    x['eng_same']=(x.pitcher_hand.astype(str)==x.batter_hand.astype(str)).astype(np.float32)
    x['eng_close']=(pd.to_numeric(x.score_diff_pitcher_team,errors='coerce').abs()<=1).astype(np.float32)
    x['eng_late']=(pd.to_numeric(x.inning,errors='coerce')>=7).astype(np.float32)
    return x
ENG_COLS=['eng_recent35','eng_recent_delta','eng_sb','eng_same','eng_close','eng_late']
ADAPTER_RAWNUM=list(dict.fromkeys(RAWNUM+[c for c in ['pitcher_team_id','batter_team_id'] if c in screen]))

def year_state(rows,year):
    selfx=build_self_state(rows.drop(columns=[TARGET],errors='ignore'))
    hist=screen.loc[screen.season.lt(int(year)),BANK_COLS]
    bank=build_global_season_bank(hist,int(year))
    gx=transform_global_season(bank,rows.drop(columns=[TARGET],errors='ignore'))
    assert len(selfx)==len(gx)==len(rows)
    return selfx.reset_index(drop=True),gx.reset_index(drop=True)

def encode_cats(source,target,cats=CAT_SMALL,prefix='code_'):
    A=pd.DataFrame(index=np.arange(len(source)));B=pd.DataFrame(index=np.arange(len(target)))
    maps={}
    for c in cats:
        s=source[c].astype('string').fillna('<NA>').astype(str);levels=pd.Index(s.unique());mp={v:i for i,v in enumerate(levels)};maps[c]=mp
        A[prefix+c]=s.map(mp).astype(np.float32);B[prefix+c]=target[c].astype('string').fillna('<NA>').astype(str).map(mp).fillna(-1).astype(np.float32)
    return A,B,maps

def exact_numeric_pair(source,target,source_year,target_year,include_cat):
    ss,gs=year_state(source,source_year);st,gt=year_state(target,target_year)
    A=pd.concat([source[RAWNUM].apply(pd.to_numeric,errors='coerce').reset_index(drop=True),ss,gs],axis=1)
    B=pd.concat([target[RAWNUM].apply(pd.to_numeric,errors='coerce').reset_index(drop=True),st,gt],axis=1)
    if include_cat:
        ca,cb,_=encode_cats(source,target);A=pd.concat([A,ca],axis=1);B=pd.concat([B,cb],axis=1)
    assert list(A.columns)==list(B.columns) and not A.columns.duplicated().any()
    return A,B

def simple_pair(source,target):
    s=add_eng(source);t=add_eng(target);cols=RAWNUM+ENG_COLS
    A=s[cols].apply(pd.to_numeric,errors='coerce').reset_index(drop=True);B=t[cols].apply(pd.to_numeric,errors='coerce').reset_index(drop=True);ca,cb,_=encode_cats(s,t);return pd.concat([A,ca],axis=1).astype(np.float32),pd.concat([B,cb],axis=1).astype(np.float32)

print('metrics/features ready')


In [ ]:
DOMAIN_MEANS=screen.groupby(['season','domain3'])[TARGET].mean().unstack()

def trailing_core_slope(source_year):
    s=DOMAIN_MEANS.loc[DOMAIN_MEANS.index<=int(source_year),'R_CORE'].dropna().tail(4)
    # Early rolling years (2019/2020) do not yet have 3 seasons of history.
    # Use zero drift with one point and a 2-point slope with two; 2022+ behavior is unchanged.
    if len(s)<2:return 0.0
    return float(np.polyfit(s.index.to_numpy(float),s.to_numpy(float),1)[0])

def fit_recent_base(source_year,target_rows,target_year,verbose=True):
    source=screen.loc[screen.season.eq(int(source_year))].reset_index(drop=True)
    target=target_rows.reset_index(drop=True)
    y=source[TARGET].to_numpy(np.float64)
    # Build exact row-state blocks once and reuse across tree/linear views.
    ss,gs=year_state(source,source_year);st,gt=year_state(target,target_year)
    A0=pd.concat([source[RAWNUM].apply(pd.to_numeric,errors='coerce').reset_index(drop=True),ss,gs],axis=1)
    B0=pd.concat([target[RAWNUM].apply(pd.to_numeric,errors='coerce').reset_index(drop=True),st,gt],axis=1)
    ca,cb,_=encode_cats(source,target);A=pd.concat([A0,ca],axis=1);B=pd.concat([B0,cb],axis=1)
    # 1) exact-ASOF binary LightGBM
    cfg=BASE_CFG['exact_lgb'];m1=lgb.LGBMClassifier(objective='binary',verbosity=-1,n_jobs=N_JOBS,**cfg);m1.fit(A,y);p1=m1.predict_proba(B)[:,1];del m1,A,B;gc.collect()
    # 2) exact-ASOF Ridge — deliberately no categorical codes, creating a different error geometry.
    A=A0.copy();B=B0.copy();del A0,B0,ss,gs,st,gt;gc.collect()
    A=A.replace([np.inf,-np.inf],np.nan);B=B.replace([np.inf,-np.inf],np.nan);med=A.median().fillna(0.);A=A.fillna(med).to_numpy(np.float32);B=B.fillna(med).to_numpy(np.float32);mu=A.mean(0);sd=A.std(0);sd=np.where(sd>1e-6,sd,1.);A=np.clip((A-mu)/sd,-8,8);B=np.clip((B-mu)/sd,-8,8)
    m2=Ridge(alpha=BASE_CFG['ridge_alpha'],solver='lsqr',max_iter=100,tol=1e-3);m2.fit(A,y);p2=np.clip(m2.predict(B),.001,.999);del m2,A,B;gc.collect()
    # 3) domain-level trend-centered LightGBM. Common drift comes only from past R_CORE labels.
    A,B=simple_pair(source,target);source_dom=source.domain3.to_numpy();target_dom=target.domain3.to_numpy();slope=trailing_core_slope(source_year)
    base=np.array([float(DOMAIN_MEANS.loc[int(source_year),d]) for d in source_dom],dtype=np.float64)
    prior={d:float(DOMAIN_MEANS.loc[int(source_year),d]+slope) for d in ['R_CORE','R_ANCHOR','F']}
    bt=np.array([prior[d] for d in target_dom],dtype=np.float64)
    cfg=BASE_CFG['trend_lgb'];m3=lgb.LGBMRegressor(objective='regression_l2',verbosity=-1,n_jobs=N_JOBS,**cfg);m3.fit(A,y-base);p3=np.clip(bt+m3.predict(B),.001,.999);del m3,A,B;gc.collect()
    p=(np.asarray(p1,float)+np.asarray(p2,float)+np.asarray(p3,float))/3.0
    meta={'source_year':int(source_year),'target_year':int(target_year),'trend_slope':slope,'trend_prior':prior}
    if verbose:print('base meta',meta)
    return np.clip(p,0,1),{'exact_lgb':p1,'exact_ridge':p2,'trend_center_lgb':p3},meta

print('base model functions ready')


In [ ]:
def fit_adapter_prep(map_rows):
    m=add_eng(map_rows.reset_index(drop=True));cols=ADAPTER_RAWNUM+ENG_COLS
    med=m[cols].apply(pd.to_numeric,errors='coerce').median().fillna(0.).to_dict();maps={}
    for c in CAT_SMALL:
        s=m[c].astype('string').fillna('<NA>').astype(str);maps[c]={v:i for i,v in enumerate(pd.Index(s.unique()))}
    return {'cols':cols,'med':med,'maps':maps}

def transform_adapter(prep,rows,base_pred):
    r=add_eng(rows.reset_index(drop=True));A=r[prep['cols']].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    for c in prep['cols']:A[c]=A[c].fillna(float(prep['med'].get(c,0.))).astype(np.float32)
    for c,mp in prep['maps'].items():A['code_'+c]=r[c].astype('string').fillna('<NA>').astype(str).map(mp).fillna(-1).astype(np.float32)
    A['base_pred']=np.asarray(base_pred,dtype=np.float32)
    assert len(A)==len(rows) and np.isfinite(A.to_numpy(float)).all()
    return A

def pitcher_half(frame,seed):
    pids=np.array(sorted(frame.pitcher_id.astype(str).unique()));rng=np.random.default_rng(int(seed));rng.shuffle(pids);A=set(pids[:len(pids)//2]);return np.where(frame.pitcher_id.astype(str).isin(A).to_numpy(),0,1).astype(np.int8)

def one_adapter_crossfit(seed,X,y,base,frame):
    fold=pitcher_half(frame,seed);corr=np.zeros(len(frame),dtype=np.float64);fold_rows=[]
    for target_fold in [0,1]:
        src=fold!=target_fold;tgt=fold==target_fold
        for d in ['R_CORE','R_ANCHOR','F']:
            tr=src & frame.domain3.eq(d).to_numpy();te=tgt & frame.domain3.eq(d).to_numpy()
            assert tr.sum()>1000 and te.sum()>500
            m=ExtraTreesRegressor(n_jobs=N_JOBS,random_state=16200+int(seed),**V14_ADAPTER_CFG);m.fit(X.loc[tr],(y-base)[tr]);corr[te]=m.predict(X.loc[te]);del m
        pp=np.clip(base[tgt]+ADAPTER_SHRINK*corr[tgt],0,1);fold_rows.append({'seed':int(seed),'target_fold':int(target_fold),'n':int(tgt.sum()),'base_bss':raw_bss(y[tgt],base[tgt]),'adapted_bss':raw_bss(y[tgt],pp),'gain':raw_bss(y[tgt],pp)-raw_bss(y[tgt],base[tgt])})
    return np.clip(base+ADAPTER_SHRINK*corr,0,1),fold_rows

print('adapter functions ready')


In [ ]:
# ===== V15 candidate feature blocks =====
def _ctx_frame(frame):
    x=frame.copy()
    x['v15_count']=x['balls_before'].astype(str)+'_'+x['strikes_before'].astype(str)
    x['v15_inning_phase']=pd.cut(pd.to_numeric(x['inning'],errors='coerce'),[-np.inf,3,6,9,np.inf],labels=['early','middle','late','extra']).astype(str)
    x['v15_pressure']=np.where((x['balls_before']==3)&(x['strikes_before']==2),'full',np.where(x['balls_before']==3,'three_ball',np.where(x['strikes_before']==2,'two_strike','normal')))
    x['v15_runners']=np.where(pd.to_numeric(x['num_runners_on'],errors='coerce').fillna(0)>0,'on','empty')
    x['v15_li_bucket']=pd.cut(pd.to_numeric(x['li'],errors='coerce'),[-np.inf,.7,1.5,np.inf],labels=['low','mid','high']).astype(str)
    return x

def _lookup(table,rows,keys,field,default=0.0):
    if len(keys)==1:
        s=table.set_index(keys[0])[field]
        return rows[keys[0]].map(s).fillna(default).to_numpy(np.float32)
    idx=pd.MultiIndex.from_frame(rows[keys])
    s=table.set_index(keys)[field]
    return s.reindex(idx).fillna(default).to_numpy(np.float32)

def _eb_table(history,keys,parent_keys=None,strength=40.0,parent_strength=80.0):
    h=history
    league=float(h[TARGET].mean())
    g=h.groupby(keys,sort=False)[TARGET].agg(['sum','size']).reset_index()
    if parent_keys is None:
        g['parent']=league
    else:
        pg=h.groupby(parent_keys,sort=False)[TARGET].agg(['sum','size']).reset_index()
        pg['parent_rate']=(pg['sum']+parent_strength*league)/(pg['size']+parent_strength)
        g=g.merge(pg[parent_keys+['parent_rate']],on=parent_keys,how='left')
        g['parent']=g['parent_rate'].fillna(league)
    g['rate']=(g['sum']+strength*g['parent'])/(g['size']+strength)
    g['reliability']=g['size']/(g['size']+strength)
    g['delta']=g['rate']-g['parent']
    g['log_n']=np.log1p(g['size'])
    return g[keys+['rate','reliability','delta','log_n']]

def _table_features(table,rows,keys,prefix,fields=('rate','reliability','delta','log_n')):
    z=pd.DataFrame(index=np.arange(len(rows)))
    for f in fields:z[prefix+'__'+f]=_lookup(table,rows,keys,f,0.0)
    return z

def build_disagreement_block(base_parts):
    p1=np.asarray(base_parts['exact_lgb'],np.float64);p2=np.asarray(base_parts['exact_ridge'],np.float64);p3=np.asarray(base_parts['trend_center_lgb'],np.float64)
    P=np.column_stack([p1,p2,p3]);z=pd.DataFrame(index=np.arange(len(p1)))
    z['dis__exact_lgb']=p1;z['dis__ridge']=p2;z['dis__trend']=p3;z['dis__std']=P.std(1);z['dis__range']=P.max(1)-P.min(1)
    z['dis__lgb_minus_ridge']=p1-p2;z['dis__lgb_minus_trend']=p1-p3;z['dis__ridge_minus_trend']=p2-p3
    for n,p in [('lgb',p1),('ridge',p2),('trend',p3)]:
        q=np.clip(p,1e-6,1-1e-6);z['dis__logit_'+n]=np.log(q/(1-q))
    return z.astype(np.float32)

def build_latest_history_blocks(source_year,target_rows,target_year):
    # Exactly one previous labeled season: 2023 -> 2024 validation, 2024 -> 2025 deployment.
    h=_ctx_frame(screen.loc[screen.season.eq(int(source_year))].reset_index(drop=True))
    r=_ctx_frame(target_rows.reset_index(drop=True))
    out={}

    # 1) exact-ASOF state that V14 base saw but V14 residual adapter did not directly see.
    selfx,gx=year_state(target_rows,target_year)
    exact=pd.concat([selfx.add_prefix('exact__'),gx.add_prefix('exact__')],axis=1)
    exact=exact.replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)
    out['EXACT_ASOF_STATE']=exact

    # 2) Cross-domain player profile: previous-season global vs same-domain ability/reliability.
    xd=[]
    for ent,tag in [('pitcher_id','p'),('batter_id','b')]:
        tg=_eb_table(h,[ent],None,80.0)
        td=_eb_table(h,[ent,'domain3'],['domain3'],60.0)
        a=_table_features(tg,r,[ent],f'xd__{tag}_global')
        b=_table_features(td,r,[ent,'domain3'],f'xd__{tag}_domain')
        b[f'xd__{tag}_domain_minus_global']=b[f'xd__{tag}_domain__rate'].to_numpy()-a[f'xd__{tag}_global__rate'].to_numpy()
        b[f'xd__{tag}_transfer_confidence']=b[f'xd__{tag}_domain_minus_global'].to_numpy()*b[f'xd__{tag}_domain__reliability'].to_numpy()
        xd += [a,b]
    out['CROSS_DOMAIN_PROFILE']=pd.concat(xd,axis=1).astype(np.float32)

    # 3) Pitcher conditional skill: conditional residual from the pitcher's own previous-season baseline.
    pc=[]
    for ctx,strength in [('v15_count',28.),('batter_hand',35.),('v15_inning_phase',35.),('base_state',35.),('v15_pressure',28.),('v15_runners',35.),('v15_li_bucket',35.)]:
        t=_eb_table(h,['pitcher_id',ctx],['pitcher_id'],strength)
        pc.append(_table_features(t,r,['pitcher_id',ctx],f'pc__{ctx}',('rate','reliability','delta')))
    out['PITCHER_CONDITIONAL']=pd.concat(pc,axis=1).astype(np.float32)

    # 4) Batter induced difficulty, smoothed toward the batter's own previous-season baseline.
    bc=[]
    for ctx,strength in [('pitcher_hand',35.),('v15_count',35.),('domain3',45.)]:
        t=_eb_table(h,['batter_id',ctx],['batter_id'],strength)
        bc.append(_table_features(t,r,['batter_id',ctx],f'bc__{ctx}',('rate','reliability','delta')))
    out['BATTER_CONDITIONAL']=pd.concat(bc,axis=1).astype(np.float32)

    # 5) Role/deployment mismatch. No target used in this block; only previous-season usage distribution.
    rg=h.groupby('pitcher_id',sort=False).agg(role_n=('inning','size'),role_inning_mean=('inning','mean'),role_inning_std=('inning','std'),role_li_mean=('li','mean'),role_li_std=('li','std'),role_late_share=('inning',lambda s:float((s>=7).mean())),role_highli_share=('li',lambda s:float((s>=1.5).mean()))).reset_index()
    rd=h.groupby(['pitcher_id','domain3'],sort=False).agg(drole_n=('inning','size'),drole_inning_mean=('inning','mean'),drole_inning_std=('inning','std'),drole_late_share=('inning',lambda s:float((s>=7).mean()))).reset_index()
    t=r[['pitcher_id','domain3','inning','li']].copy();t['_i']=np.arange(len(t));t=t.merge(rg,on='pitcher_id',how='left').merge(rd,on=['pitcher_id','domain3'],how='left').sort_values('_i')
    role=pd.DataFrame(index=np.arange(len(r)))
    im=t.role_inning_mean.fillna(float(h.inning.mean()));isd=t.role_inning_std.fillna(2.).clip(lower=.5);lm=t.role_li_mean.fillna(float(h.li.mean()));ls=t.role_li_std.fillna(1.).clip(lower=.2)
    role['role__log_n']=np.log1p(t.role_n.fillna(0));role['role__inning_z']=(t.inning-im)/isd;role['role__inning_abs_z']=np.abs(role['role__inning_z']);role['role__li_z']=(t.li-lm)/ls;role['role__li_abs_z']=np.abs(role['role__li_z'])
    role['role__late_surprise']=(t.inning>=7).astype(float)-t.role_late_share.fillna(.3);role['role__highli_surprise']=(t.li>=1.5).astype(float)-t.role_highli_share.fillna(.2)
    dim=t.drole_inning_mean.fillna(im);disd=t.drole_inning_std.fillna(isd).clip(lower=.5);role['role__domain_log_n']=np.log1p(t.drole_n.fillna(0));role['role__domain_inning_abs_z']=np.abs((t.inning-dim)/disd);role['role__domain_late_surprise']=(t.inning>=7).astype(float)-t.drole_late_share.fillna(t.role_late_share).fillna(.3)
    out['ROLE_DEPLOYMENT']=role.replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)

    for k,v in out.items():
        assert len(v)==len(target_rows) and not v.columns.duplicated().any() and np.isfinite(v.to_numpy(float)).all(),k
    return out

print('V15 feature block builders ready')


In [ ]:
# ===== V18 STABLE TEMPORAL FEATURE FAMILY =====
# Signal discovery principle:
#  - use ONLY historical labeled seasons strictly before the row's season
#  - estimate a pitcher's relative sensitivity to batter hand and pressure×batter-hand
#  - shrink child context to a pitcher parent, then expose delta/reliability
#  - add deterministic count indicators so the linear residual model can separate league count difficulty
#  - no current test aggregation, no row_id, no target-year target lookup

def _v18_pressure_frame(frame):
    r=frame.copy()
    balls=pd.to_numeric(r['balls_before'],errors='coerce').fillna(0).astype(int)
    strikes=pd.to_numeric(r['strikes_before'],errors='coerce').fillna(0).astype(int)
    r['v18_pressure']=np.where((balls==3)&(strikes==2),'full',np.where(balls==3,'3ball',np.where(strikes==2,'2strike','normal')))
    r['v18_pressure_bhand']=r['v18_pressure'].astype(str)+'|'+r['batter_hand'].astype(str)
    return r

def _v18_weighted_conditional_table(history,ctx,strength,cutoff_year):
    h=_v18_pressure_frame(history.reset_index(drop=True))
    # explicit temporal decay: most recent labeled season weight=1
    h['_w']=np.power(V18_DECAY,np.maximum(int(cutoff_year)-1-h['season'].astype(int),0)).astype(np.float64)
    h['_wy']=h['_w']*pd.to_numeric(h[TARGET],errors='coerce').to_numpy(np.float64)
    mu=float(np.average(h[TARGET].to_numpy(np.float64),weights=h['_w']))
    pg=h.groupby('pitcher_id',sort=False).agg(sw=('_w','sum'),sy=('_wy','sum')).reset_index()
    pg['parent_rate']=(pg['sy']+V18_PARENT_STRENGTH*mu)/(pg['sw']+V18_PARENT_STRENGTH)
    g=h.groupby(['pitcher_id',ctx],sort=False).agg(sw=('_w','sum'),sy=('_wy','sum')).reset_index().merge(pg[['pitcher_id','parent_rate']],on='pitcher_id',how='left')
    g['rate']=(g['sy']+float(strength)*g['parent_rate'])/(g['sw']+float(strength))
    g['delta']=g['rate']-g['parent_rate']
    g['reliability']=g['sw']/(g['sw']+float(strength))
    return g[['pitcher_id',ctx,'delta','reliability']]

def build_v18_stable_specs(cutoff_year):
    lo=max(int(screen.season.min()),int(cutoff_year)-V18_HISTORY_YEARS)
    h=screen.loc[screen.season.lt(int(cutoff_year)) & screen.season.ge(lo)].reset_index(drop=True)
    if h.empty: raise RuntimeError(f'no stable history for cutoff {cutoff_year}')
    specs=[]
    for ctx,strength,prefix in [
        ('batter_hand',V18_HAND_STRENGTH,'stable__batter_hand'),
        ('v18_pressure_bhand',V18_PRESSURE_HAND_STRENGTH,'stable__pressure_bhand'),
    ]:
        specs.append({'ctx':ctx,'prefix':prefix,'table':_v18_weighted_conditional_table(h,ctx,strength,cutoff_year)})
    return specs

def transform_v18_stable_specs(specs,rows):
    r=_v18_pressure_frame(rows.reset_index(drop=True));z=pd.DataFrame(index=np.arange(len(r)))
    for spec in specs:
        de=_lookup(spec['table'],r,['pitcher_id',spec['ctx']],'delta',0.0).astype(np.float32)
        re=_lookup(spec['table'],r,['pitcher_id',spec['ctx']],'reliability',0.0).astype(np.float32)
        z[spec['prefix']+'__delta']=de
        z[spec['prefix']+'__weighted_delta']=(de*re).astype(np.float32)
        z[spec['prefix']+'__reliability']=re
    balls=pd.to_numeric(r['balls_before'],errors='coerce').fillna(0).astype(int).to_numpy()
    strikes=pd.to_numeric(r['strikes_before'],errors='coerce').fillna(0).astype(int).to_numpy()
    for b in range(4): z[f'ctx__ball_{b}']=(balls==b).astype(np.float32)
    for s in range(3): z[f'ctx__strike_{s}']=(strikes==s).astype(np.float32)
    for b in range(4):
        for s in range(3): z[f'ctx__count_{b}_{s}']=((balls==b)&(strikes==s)).astype(np.float32)
    z=z.replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)
    assert len(z)==len(rows) and not z.columns.duplicated().any() and np.isfinite(z.to_numpy(float)).all()
    return z

def v18_stable_matrix(year):
    rows=screen.loc[screen.season.eq(int(year))].reset_index(drop=True)
    specs=build_v18_stable_specs(int(year))
    return transform_v18_stable_specs(specs,rows),rows,specs

print('V18 stable temporal feature functions ready')


## V20 hypothesis — why role/usage is worth one clean test

Earlier competition-only feature work repeatedly found `season_workload_vs_prev` and `role_inning_z/abs_z` useful enough to survive feature pruning, but most of those experiments evaluated them inside same-year residual specialists. V20 changes the question:

> Does a pitcher's **preseason expected usage/role**, compared with the current row's independently reconstructed workload pace and deployment context, explain V18 residuals in a way that transfers to the next season?

The feature family intentionally excludes batter-heavy conditionals, exact matchup, month target statistics, same-year disagreement, and TrackMan. Month is used only as a deterministic progress coordinate to normalize workload pace; no target rate is conditioned on month.

V19's low-gamma player-state result is recorded as a weak positive specialist, but is not blended into V20 selection so the experiment remains attributable.

In [ ]:
print('===== V18 STAGE B: FULL-DATA SIGNAL PERSISTENCE AUDIT =====')

def _year_cond_effect(year,ctx,strength):
    h=_v18_pressure_frame(screen.loc[screen.season.eq(int(year))].reset_index(drop=True))
    league=float(h[TARGET].mean())
    pg=h.groupby('pitcher_id',sort=False)[TARGET].agg(['sum','size']).reset_index()
    pg['pr']=(pg['sum']+100.*league)/(pg['size']+100.)
    g=h.groupby(['pitcher_id',ctx],sort=False)[TARGET].agg(['sum','size']).reset_index().merge(pg[['pitcher_id','pr']],on='pitcher_id',how='left')
    g['rate']=(g['sum']+strength*g['pr'])/(g['size']+strength);g['delta']=g['rate']-g['pr'];g['rel']=g['size']/(g['size']+strength)
    return g[['pitcher_id',ctx,'delta','rel']]

persist=[]
for ctx,strength in [('batter_hand',35.),('v18_pressure_bhand',45.)]:
    for y0,y1 in zip(sorted(screen.season.unique())[:-1],sorted(screen.season.unique())[1:]):
        a=_year_cond_effect(int(y0),ctx,strength);b=_year_cond_effect(int(y1),ctx,strength)
        m=a.merge(b,on=['pitcher_id',ctx],suffixes=('_0','_1'))
        ok=(m.rel_0>.10)&(m.rel_1>.10)
        corr=float(np.corrcoef(m.loc[ok,'delta_0'],m.loc[ok,'delta_1'])[0,1]) if ok.sum()>20 else np.nan
        persist.append({'context':ctx,'source':int(y0),'target':int(y1),'groups':int(ok.sum()),'corr':corr})
persist_df=pd.DataFrame(persist)
display(persist_df)
print('\nPersistence summary:');display(persist_df.groupby('context')['corr'].agg(['mean','min','max']))
persist_df.to_csv(REPORT_DIR/'v18_conditional_persistence.csv',index=False)
print('AUDIT_NOTE: batter-hand and pressure×batter-hand are intentionally retained; weaker count/month/matchup/TrackMan-style candidates are not promoted into the deploy model.')


In [ ]:
print('===== V20 STAGE C: BUILD RECENT ROLLING BASES 2019→20 ... 2023→24 =====')
BASE_BY_YEAR={};BASE_PARTS={};ROWS={};Y={}
for target_year in [2020,2021,2022,2023,2024]:
    rows=screen.loc[screen.season.eq(target_year)].reset_index(drop=True)
    t0=time.time();p,parts,meta=fit_recent_base(target_year-1,rows,target_year,verbose=False)
    BASE_BY_YEAR[target_year]=p;BASE_PARTS[target_year]=parts;ROWS[target_year]=rows;Y[target_year]=rows[TARGET].to_numpy(np.float64)
    print({'target_year':target_year,'source_year':target_year-1,'base_bss':raw_bss(Y[target_year],p),'pred_mean':float(p.mean()),'target_mean':float(Y[target_year].mean()),'sec':time.time()-t0})
print('ROLLING_BASES_READY')


In [ ]:
def one_adapter_crossfit_cfg(seed,X,y,base,frame,cfg):
    fold=pitcher_half(frame,seed);corr=np.zeros(len(frame),dtype=np.float64);fold_rows=[]
    for target_fold in [0,1]:
        src=fold!=target_fold;tgt=fold==target_fold
        for d in ['R_CORE','R_ANCHOR','F']:
            tr=src & frame.domain3.eq(d).to_numpy();te=tgt & frame.domain3.eq(d).to_numpy()
            min_tr=1000 if not SMOKE_MODE else 80;min_te=500 if not SMOKE_MODE else 40
            assert tr.sum()>min_tr and te.sum()>min_te,(d,tr.sum(),te.sum())
            m=ExtraTreesRegressor(n_jobs=N_JOBS,random_state=16200+int(seed),**cfg)
            m.fit(X.loc[tr],(y-base)[tr]);corr[te]=m.predict(X.loc[te]);del m
        pp=np.clip(base[tgt]+ADAPTER_SHRINK*corr[tgt],0,1)
        fold_rows.append({'seed':int(seed),'target_fold':int(target_fold),'n':int(tgt.sum()),'base_bss':raw_bss(y[tgt],base[tgt]),'adapted_bss':raw_bss(y[tgt],pp),'gain':raw_bss(y[tgt],pp)-raw_bss(y[tgt],base[tgt])})
    return np.clip(base+ADAPTER_SHRINK*corr,0,1),fold_rows

def score_prediction(label,p,reference=None):
    row={'name':label,'bss':raw_bss(y24,p),'gain_vs_base':raw_bss(y24,p)-raw_bss(y24,p_base24)}
    if reference is not None:row['gain_vs_reference']=raw_bss(y24,p)-raw_bss(y24,reference)
    for d in ['R_CORE','R_ANCHOR','F']:
        z=raw24.domain3.eq(d).to_numpy();row[d+'_bss']=raw_bss(y24[z],p[z])
        if reference is not None:row[d+'_gain_vs_reference']=raw_bss(y24[z],p[z])-raw_bss(y24[z],reference[z])
    return row

def assemble_blocks(names):
    parts=[X_V14]
    for n in names:parts.append(BLOCKS_24[n].reset_index(drop=True))
    X=pd.concat(parts,axis=1)
    assert not X.columns.duplicated().any() and np.isfinite(X.to_numpy(float)).all()
    return X

print('V18 V15-crossfit helpers ready')


In [ ]:
print('===== V20 STAGE D: REBUILD V15 OOF VIEW FOR 2020/2021/2022/2023/2024 =====')

def build_v15_matrix_for_year(target_year):
    source_year=int(target_year)-1;rows=ROWS[target_year];base=BASE_BY_YEAR[target_year]
    map_rows=screen.loc[screen.season.eq(source_year)].reset_index(drop=True)
    prep=fit_adapter_prep(map_rows);x0=transform_adapter(prep,rows,base)
    blocks=build_latest_history_blocks(source_year,rows,target_year)
    X=pd.concat([x0.reset_index(drop=True),blocks['PITCHER_CONDITIONAL'].reset_index(drop=True),blocks['EXACT_ASOF_STATE'].reset_index(drop=True)],axis=1)
    assert not X.columns.duplicated().any() and np.isfinite(X.to_numpy(float)).all()
    return X

def v15_oof_year(target_year,X):
    frame=ROWS[target_year];yy=Y[target_year];base=BASE_BY_YEAR[target_year];preds=[];seed_rows=[]
    for seed in CF_SEEDS:
        p,_=one_adapter_crossfit_cfg(seed,X,yy,base,frame,V15_CF_CFG)
        preds.append(p);seed_rows.append({'year':target_year,'seed':int(seed),'bss':raw_bss(yy,p),'gain_vs_base':raw_bss(yy,p)-raw_bss(yy,base)})
        print(seed_rows[-1]);gc.collect()
    consensus=np.mean(np.column_stack(preds),axis=1)
    print('V15 OOF CONSENSUS',target_year,raw_bss(yy,consensus))
    return consensus,seed_rows

V15_OOF={};V15_SEED_ROWS=[]
for y in [2020,2021,2022,2023,2024]:
    X=build_v15_matrix_for_year(y);print('V15 matrix',y,X.shape)
    V15_OOF[y],rr=v15_oof_year(y,X);V15_SEED_ROWS.extend(rr)
    del X;gc.collect()
pd.DataFrame(V15_SEED_ROWS).to_csv(REPORT_DIR/'v20_v15_oof_seed_scores.csv',index=False)
print('V15_OOF_READY')


In [ ]:
print('===== V20 STAGE E: REBUILD V18 STRICT NEXT-YEAR BASELINE 2020→21 ... 2023→24 =====')
STABLE={};STABLE_SPECS={}
for y in [2020,2021,2022,2023,2024]:
    STABLE[y],_,STABLE_SPECS[y]=v18_stable_matrix(y)
    print('stable matrix',y,STABLE[y].shape)

TEMPORAL_MODELS={};TEMPORAL_ROWS=[];P18={};CORR={}
for target_year in [2021,2022,2023,2024]:
    source_year=target_year-1
    # Crucial: correction is trained on source-year V15 OOF residual, never on target-year labels.
    model=Ridge(alpha=V18_RIDGE_ALPHA,solver='lsqr',max_iter=200,tol=1e-5)
    # explicit standardization, persisted as simple arrays for reproducibility
    A=STABLE[source_year].to_numpy(np.float32);B=STABLE[target_year].to_numpy(np.float32)
    mu=A.mean(0);sd=A.std(0);sd=np.where(sd>1e-6,sd,1.).astype(np.float32)
    Az=np.clip((A-mu)/sd,-8,8);Bz=np.clip((B-mu)/sd,-8,8)
    resid=Y[source_year]-V15_OOF[source_year]
    model.fit(Az,resid);corr=np.asarray(model.predict(Bz),np.float64);CORR[target_year]=corr
    TEMPORAL_MODELS[target_year]={'model':model,'mu':mu.astype(np.float32),'sd':sd,'source_year':source_year}
    base=V15_OOF[target_year];frame=ROWS[target_year];core=frame.domain3.eq(V18_ROUTE_DOMAIN).to_numpy()
    p=base.copy();p[core]=np.clip(p[core]+V18_GAMMA*corr[core],0,1);P18[target_year]=p
    row={'source_year':source_year,'target_year':target_year,'v15_bss':raw_bss(Y[target_year],base),'v18_bss':raw_bss(Y[target_year],p),'gain':raw_bss(Y[target_year],p)-raw_bss(Y[target_year],base),'core_gain':raw_bss(Y[target_year][core],p[core])-raw_bss(Y[target_year][core],base[core])}
    TEMPORAL_ROWS.append(row);print(row)
    print('gamma stress',target_year,{g:raw_bss(Y[target_year],np.where(core,np.clip(base+g*corr,0,1),base))-raw_bss(Y[target_year],base) for g in V18_GAMMA_STRESS})

temporal_df=pd.DataFrame(TEMPORAL_ROWS);display(temporal_df);temporal_df.to_csv(REPORT_DIR/'v20_v18_strict_temporal_transfer.csv',index=False)
print('V18_ROLLING_BASELINE_READY')


In [ ]:
print('===== V20 STAGE F: RECONFIRM V18 CHAMPION BASELINE =====')
p15=V15_OOF[2024];p18=P18[2024];frame=ROWS[2024];yy=Y[2024]
boot=pitcher_cluster_bootstrap_gain(frame,yy,p18,p15,draws=3000 if not SMOKE_MODE else 200,seed=18018)
print('bootstrap V18 vs V15:',boot)
month_rows=[]
for mo in sorted(frame.game_month.unique()):
    z=frame.game_month.eq(mo).to_numpy();month_rows.append({'month':int(mo),'n':int(z.sum()),'v15':raw_bss(yy[z],p15[z]),'v18':raw_bss(yy[z],p18[z]),'gain':raw_bss(yy[z],p18[z])-raw_bss(yy[z],p15[z])})
month_df=pd.DataFrame(month_rows);display(month_df);month_df.to_csv(REPORT_DIR/'v20_v18_months.csv',index=False)
early=frame.game_month.le(6).to_numpy();late=frame.game_month.ge(7).to_numpy();early_gain=raw_bss(yy[early],p18[early])-raw_bss(yy[early],p15[early]);late_gain=raw_bss(yy[late],p18[late])-raw_bss(yy[late],p15[late])
print('early_gain',early_gain,'late_gain',late_gain)

domain_rows=[]
for d in ['R_CORE','R_ANCHOR','F']:
    z=frame.domain3.eq(d).to_numpy();domain_rows.append({'domain':d,'v15':raw_bss(yy[z],p15[z]),'v18':raw_bss(yy[z],p18[z]),'gain':raw_bss(yy[z],p18[z])-raw_bss(yy[z],p15[z])})
domain_df=pd.DataFrame(domain_rows);display(domain_df)

g23=float(temporal_df.loc[temporal_df.target_year.eq(2023),'gain'].iloc[0]);g24=float(temporal_df.loc[temporal_df.target_year.eq(2024),'gain'].iloc[0]);core24=float(domain_df.loc[domain_df.domain.eq('R_CORE'),'gain'].iloc[0])
stress_ok=True
for ty in [2023,2024]:
    base=V15_OOF[ty];corr=CORR[ty];core=ROWS[ty].domain3.eq(V18_ROUTE_DOMAIN).to_numpy()
    for g in (0.65,0.75,0.85):
        pp=base.copy();pp[core]=np.clip(pp[core]+g*corr[core],0,1)
        stress_ok &= (raw_bss(Y[ty],pp)-raw_bss(Y[ty],base)>0)
V18_SUBMIT_RECOMMENDED=bool(
    g23>=MIN_GAIN_2023 and g24>=MIN_GAIN_2024 and core24>=MIN_CORE_GAIN_2024 and
    boot['p05']>=MIN_BOOT_P05 and boot['prob_positive']>=MIN_BOOT_PROB and late_gain>=MIN_LATE_GAIN and stress_ok
)
summary={
 'protocol':PROTOCOL_VERSION,'config_hash':CONFIG_HASH,'current_lb':CURRENT_LB,'target_lb':TARGET_LB,
 'temporal_2022_to_2023_gain':g23,'temporal_2023_to_2024_gain':g24,'core_2024_gain':core24,
 'bootstrap':boot,'early_gain':float(early_gain),'late_gain':float(late_gain),'gamma':V18_GAMMA,
 'route_domain':V18_ROUTE_DOMAIN,'submit_recommended':V18_SUBMIT_RECOMMENDED,
 'warning':'2023/2024 are development-exposed. The key promotion evidence is strict source-year residual transfer plus stable adjacent-year feature persistence; this is not a 1100 guarantee.'
}
print(json.dumps(summary,indent=2,ensure_ascii=False))
print('V18_SUBMIT_RECOMMENDED:',V18_SUBMIT_RECOMMENDED)
(REPORT_DIR/'v20_v18_baseline_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')


In [ ]:
# ===== V20 STAGE J1: BUILD VERIFIED V18 CHAMPION PACKAGE BASE =====
import shutil, zipfile, subprocess, sys, joblib, sklearn

# Reuse the exact 2024 objects already built in the rolling validation.
raw24=ROWS[2024].reset_index(drop=True)
y24=Y[2024].copy()
p_base24=BASE_BY_YEAR[2024].copy()
BLOCKS_24=build_latest_history_blocks(BASE_SOURCE_YEAR,raw24,VALID_YEAR)

# This is intentionally a challenger submission even though the strict V15 promotion gate was False.
# The candidate is the exact feature choice measured above: PITCHER_CONDITIONAL + EXACT_ASOF_STATE.
DEPLOY_SELECTED_BLOCKS=['PITCHER_CONDITIONAL','EXACT_ASOF_STATE']
assert set(DEPLOY_SELECTED_BLOCKS).issubset(set(BLOCKS_24)), DEPLOY_SELECTED_BLOCKS
print('V18_SUBMIT_RECOMMENDED:',V18_SUBMIT_RECOMMENDED)
print('DEPLOY_SELECTED_BLOCKS:',DEPLOY_SELECTED_BLOCKS)
print('V15 LB champion remains the external fallback. V18 correction is enabled in the package only when the temporal gate passes.')

TEST_PATH=discover_csv({'row_id','season','pitcher_id','batter_id','game_type','pitcher_team_id','batter_team_id'},'test.csv')
if TEST_PATH is None: raise FileNotFoundError('official test.csv not found')
test=pd.read_csv(TEST_PATH,encoding='utf-8-sig',low_memory=False)
test['season']=pd.to_numeric(test['season'],errors='raise').astype(int)
test['game_type']=test['game_type'].astype('string').fillna('<NA>').astype(str)
assert TARGET not in test
expected_year=int(screen.season.max())+1
if bool((test.season!=expected_year).any()): raise RuntimeError(f'evaluation season differs from train-derived expected year {expected_year}')
if bool((~test.game_type.isin(['R','F'])).any()): raise RuntimeError('unexpected game_type')
test['domain3']=np.where(test.game_type.eq('F'),'F',np.where((test.pitcher_team_id==ANCHOR_TEAM_ID)|(test.batter_team_id==ANCHOR_TEAM_ID),'R_ANCHOR','R_CORE'))
print('DEPLOY expected year:',expected_year,'format-test rows:',len(test))

WORK_ROOT=Path('/kaggle/working/v20_temporal_role_usage_deploy') if Path('/kaggle/working').exists() else Path('./v20_temporal_role_usage_deploy')
MODEL_DIR=WORK_ROOT/'model'
if WORK_ROOT.exists(): shutil.rmtree(WORK_ROOT)
MODEL_DIR.mkdir(parents=True,exist_ok=True)

# ---------- 1) 2024 -> 2025 shared base; persist exact inference artifacts ----------
source=raw24.reset_index(drop=True); y_source=y24.copy()
ss=build_self_state(source.drop(columns=[TARGET],errors='ignore'))
source_bank=build_global_season_bank(screen.loc[screen.season.lt(VALID_YEAR),BANK_COLS],VALID_YEAR)
gs=transform_global_season(source_bank,source.drop(columns=[TARGET],errors='ignore'))
target_bank=build_global_season_bank(screen.loc[screen.season.lt(expected_year),BANK_COLS],expected_year)
st=build_self_state(test); gt=transform_global_season(target_bank,test)
A0=pd.concat([source[RAWNUM].apply(pd.to_numeric,errors='coerce').reset_index(drop=True),ss.reset_index(drop=True),gs.reset_index(drop=True)],axis=1)
B0=pd.concat([test[RAWNUM].apply(pd.to_numeric,errors='coerce').reset_index(drop=True),st.reset_index(drop=True),gt.reset_index(drop=True)],axis=1)

ca,cb,exact_maps=encode_cats(source,test)
A_exact=pd.concat([A0,ca],axis=1);B_exact=pd.concat([B0,cb],axis=1)
m1=lgb.LGBMClassifier(objective='binary',verbosity=-1,n_jobs=N_JOBS,**BASE_CFG['exact_lgb']);m1.fit(A_exact,y_source);p1=m1.predict_proba(B_exact)[:,1]
joblib.dump(m1,MODEL_DIR/'base_exact_lgb.joblib',compress=3); exact_feature_cols=list(A_exact.columns)
del m1,A_exact,B_exact;gc.collect()

Ar=A0.replace([np.inf,-np.inf],np.nan);Br=B0.replace([np.inf,-np.inf],np.nan)
ridge_med=Ar.median().fillna(0.);Ar=Ar.fillna(ridge_med).to_numpy(np.float32);Br=Br.fillna(ridge_med).to_numpy(np.float32)
ridge_mu=Ar.mean(0);ridge_sd=Ar.std(0);ridge_sd=np.where(ridge_sd>1e-6,ridge_sd,1.);Ar=np.clip((Ar-ridge_mu)/ridge_sd,-8,8);Br=np.clip((Br-ridge_mu)/ridge_sd,-8,8)
m2=Ridge(alpha=BASE_CFG['ridge_alpha'],solver='lsqr',max_iter=100,tol=1e-3);m2.fit(Ar,y_source);p2=np.clip(m2.predict(Br),.001,.999)
joblib.dump(m2,MODEL_DIR/'base_ridge.joblib',compress=3); ridge_feature_cols=list(A0.columns)
del m2,Ar,Br;gc.collect()

s=add_eng(source);t=add_eng(test);trend_num_cols=RAWNUM+ENG_COLS
An=s[trend_num_cols].apply(pd.to_numeric,errors='coerce').reset_index(drop=True);Bn=t[trend_num_cols].apply(pd.to_numeric,errors='coerce').reset_index(drop=True)
cta,ctb,trend_maps=encode_cats(s,t);At=pd.concat([An,cta],axis=1).astype(np.float32);Bt=pd.concat([Bn,ctb],axis=1).astype(np.float32)
slope=trailing_core_slope(VALID_YEAR);source_dom=source.domain3.to_numpy();base_dom=np.array([float(DOMAIN_MEANS.loc[VALID_YEAR,d]) for d in source_dom],dtype=np.float64)
trend_prior={d:float(DOMAIN_MEANS.loc[VALID_YEAR,d]+slope) for d in ['R_CORE','R_ANCHOR','F']};bt=np.array([trend_prior[d] for d in test.domain3.to_numpy()],dtype=np.float64)
m3=lgb.LGBMRegressor(objective='regression_l2',verbosity=-1,n_jobs=N_JOBS,**BASE_CFG['trend_lgb']);m3.fit(At,y_source-base_dom);p3=np.clip(bt+m3.predict(Bt),.001,.999)
joblib.dump(m3,MODEL_DIR/'base_trend_lgb.joblib',compress=3); trend_feature_cols=list(At.columns)
del m3,At,Bt,An,Bn;gc.collect()

p_base25=np.clip((np.asarray(p1,float)+np.asarray(p2,float)+np.asarray(p3,float))/3.0,0,1)
print('DEPLOY base format-test pred mean:',float(p_base25.mean()))

# ---------- 2) V15 selected feature banks: 2023->2024 train semantics, 2024->2025 frozen inference semantics ----------
PC_CONTEXTS=[('v15_count',28.),('batter_hand',35.),('v15_inning_phase',35.),('base_state',35.),('v15_pressure',28.),('v15_runners',35.),('v15_li_bucket',35.)]

def make_pc_specs(source_year):
    h=_ctx_frame(screen.loc[screen.season.eq(int(source_year))].reset_index(drop=True)); specs=[]
    for ctx,strength in PC_CONTEXTS:
        table=_eb_table(h,['pitcher_id',ctx],['pitcher_id'],strength)
        specs.append({'ctx':ctx,'prefix':'pc__'+ctx,'table':table})
    return specs

def transform_pc_specs(specs,rows):
    r=_ctx_frame(rows.reset_index(drop=True));parts=[]
    for spec in specs:
        z=_table_features(spec['table'],r,['pitcher_id',spec['ctx']],spec['prefix'],('rate','reliability','delta'));parts.append(z)
    return pd.concat(parts,axis=1).astype(np.float32)

def exact_adapter_block(rows,year):
    sx,gx=year_state(rows,year);out=pd.concat([sx.add_prefix('exact__'),gx.add_prefix('exact__')],axis=1)
    return out.replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)

pc23=make_pc_specs(BASE_SOURCE_YEAR);pc24=make_pc_specs(VALID_YEAR)
pc24_train=transform_pc_specs(pc23,raw24);pc25=transform_pc_specs(pc24,test)
ex24=exact_adapter_block(raw24,VALID_YEAR);ex25=pd.concat([st.add_prefix('exact__'),gt.add_prefix('exact__')],axis=1).replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)
# Verify deployment training features are exactly the already-validated V15 blocks.
assert list(pc24_train.columns)==list(BLOCKS_24['PITCHER_CONDITIONAL'].columns)
assert np.max(np.abs(pc24_train.to_numpy(float)-BLOCKS_24['PITCHER_CONDITIONAL'].to_numpy(float)))<1e-7
assert list(ex24.columns)==list(BLOCKS_24['EXACT_ASOF_STATE'].columns)
assert np.max(np.abs(ex24.to_numpy(float)-BLOCKS_24['EXACT_ASOF_STATE'].to_numpy(float)))<1e-7

prep24=fit_adapter_prep(raw24)
X24_base=transform_adapter(prep24,raw24,p_base24); X25_base=transform_adapter(prep24,test,p_base25)
adapter_base_feature_cols=list(X24_base.columns)
X24_final=pd.concat([X24_base.reset_index(drop=True),pc24_train.reset_index(drop=True),ex24.reset_index(drop=True)],axis=1)
X25=pd.concat([X25_base.reset_index(drop=True),pc25.reset_index(drop=True),ex25.reset_index(drop=True)],axis=1)
assert list(X24_final.columns)==list(X25.columns) and not X24_final.columns.duplicated().any()
assert np.isfinite(X24_final.to_numpy(float)).all() and np.isfinite(X25.to_numpy(float)).all()
adapter_feature_cols=list(X24_final.columns); pitcher_conditional_cols=list(pc25.columns); exact_adapter_cols=list(ex25.columns)
print('V15 deploy adapter features:',X24_final.shape,'-> test',X25.shape)

# ---------- 3) final residual adapters from ALL labeled 2024 ----------
res24=y24-p_base24;corrections=[]
for seed in CF_SEEDS:
    c=np.zeros(len(test),dtype=np.float64)
    for d in ['R_CORE','R_ANCHOR','F']:
        tr=raw24.domain3.eq(d).to_numpy();te=test.domain3.eq(d).to_numpy()
        m=ExtraTreesRegressor(n_jobs=N_JOBS,random_state=16200+int(seed),**FINAL_ADAPTER_CFG);m.fit(X24_final.loc[tr],res24[tr])
        joblib.dump(m,MODEL_DIR/f'adapter_seed{int(seed)}_{d}.joblib',compress=3)
        if te.any():c[te]=m.predict(X25.loc[te])
        del m
    corrections.append(c)
correction=np.mean(np.column_stack(corrections),axis=1)
final_v15_direct=np.clip(p_base25+ADAPTER_SHRINK*correction,0,1)
final_direct=final_v15_direct.copy()
assert np.isfinite(final_direct).all() and ((final_direct>=0)&(final_direct<=1)).all()

preprocess={
    'protocol':'V15_CHALLENGER_DEPLOY_PITCHER_CONDITIONAL_EXACT_ASOF', 'source_protocol':PROTOCOL_VERSION, 'config_hash':CONFIG_HASH,
    'expected_year':expected_year,'anchor_team_id':ANCHOR_TEAM_ID,'selected_blocks':DEPLOY_SELECTED_BLOCKS,
    'rawnum':RAWNUM,'cat_small':CAT_SMALL,'eng_cols':ENG_COLS,'adapter_rawnum':ADAPTER_RAWNUM,
    'target_global_bank':target_bank,
    'exact_cat_maps':exact_maps,'exact_feature_cols':exact_feature_cols,
    'ridge_feature_cols':ridge_feature_cols,'ridge_med':ridge_med.to_dict(),'ridge_mu':ridge_mu.astype(np.float32),'ridge_sd':ridge_sd.astype(np.float32),
    'trend_num_cols':trend_num_cols,'trend_cat_maps':trend_maps,'trend_feature_cols':trend_feature_cols,'trend_prior':trend_prior,
    'adapter_prep':prep24,'adapter_base_feature_cols':adapter_base_feature_cols,'adapter_feature_cols':adapter_feature_cols,
    'adapter_seeds':list(CF_SEEDS),'adapter_shrink':float(ADAPTER_SHRINK),
    'pitcher_conditional_specs':pc24,'pitcher_conditional_cols':pitcher_conditional_cols,'exact_adapter_cols':exact_adapter_cols,
    'validation_summary':summary, 'challenger_note':'Strict V15 promotion gate was false; packaged by explicit user request for one leaderboard challenger test.'
}
# preprocess is extended below with V18 temporal artifacts before dumping


# ---------- 4) fit strict-temporal correction on 2024 V15 OOF residual; apply to 2025 R_CORE only ----------
# Re-use the already computed 2024 OOF consensus from Stage D. This prevents fitting the correction to an in-sample V15 prediction.
assert 2024 in V15_OOF and len(V15_OOF[2024])==len(raw24)
stable24,_,stable_specs24=v18_stable_matrix(2024)
stable_specs25=build_v18_stable_specs(expected_year)
stable25=transform_v18_stable_specs(stable_specs25,test)
assert list(stable24.columns)==list(stable25.columns)
A=stable24.to_numpy(np.float32);B=stable25.to_numpy(np.float32)
mu=A.mean(0);sd=A.std(0);sd=np.where(sd>1e-6,sd,1.).astype(np.float32)
Az=np.clip((A-mu)/sd,-8,8);Bz=np.clip((B-mu)/sd,-8,8)
stable_model=Ridge(alpha=V18_RIDGE_ALPHA,solver='lsqr',max_iter=200,tol=1e-5)
stable_model.fit(Az,y24-V15_OOF[2024])
joblib.dump(stable_model,MODEL_DIR/'v18_stable_ridge.joblib',compress=3)
stable_corr25=np.asarray(stable_model.predict(Bz),np.float64)
if V18_SUBMIT_RECOMMENDED:
    core25=test.domain3.eq(V18_ROUTE_DOMAIN).to_numpy()
    final_direct[core25]=np.clip(final_direct[core25]+V18_GAMMA*stable_corr25[core25],0,1)
print('V18 deploy correction enabled:',V18_SUBMIT_RECOMMENDED,'stable features:',stable24.shape,'format-test core rows:',int(test.domain3.eq(V18_ROUTE_DOMAIN).sum()))

preprocess.update({
    'protocol':'V18_TEMPORAL_STABLE_HAND_PRESSURE_COUNT_RIDGE',
    'source_protocol':PROTOCOL_VERSION,
    'validation_summary':summary,
    'v18_stable_enabled':bool(V18_SUBMIT_RECOMMENDED),
    'v18_stable_specs':stable_specs25,
    'v18_stable_cols':list(stable25.columns),
    'v18_stable_mu':mu.astype(np.float32),
    'v18_stable_sd':sd.astype(np.float32),
    'v18_stable_gamma':float(V18_GAMMA),
    'v18_route_domain':V18_ROUTE_DOMAIN,
    'v18_stable_model_file':'v18_stable_ridge.joblib',
    'challenger_note':'V18 adds only a strict next-year, OOF-residual-trained, low-dimensional stable conditional Ridge correction to V15; R_ANCHOR/F are unchanged.'
})
joblib.dump(preprocess,MODEL_DIR/'preprocess.joblib',compress=3)
assert np.isfinite(final_direct).all() and ((final_direct>=0)&(final_direct<=1)).all()

# ---------- 5) server runtime ----------
SCRIPT_SOURCE="from pathlib import Path\nimport sys\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nROOT=Path(__file__).resolve().parent\nMODEL=ROOT/'model'\nDATA=ROOT/'data'\nOUT=ROOT/'output'\nOUT.mkdir(exist_ok=True)\nMETA=joblib.load(MODEL/'preprocess.joblib')\nTARGET='control_success'\n\n# Row-wise features: only official current-row inputs. No target aggregation and no other prediction/test row is used.\ndef _n(frame,c,default=np.nan):\n    if c in frame:return pd.to_numeric(frame[c],errors='coerce').to_numpy(np.float64)\n    return np.full(len(frame),default,np.float64)\n\ndef build_self_state(frame):\n    n=len(frame);z=pd.DataFrame(index=np.arange(n))\n    pn=np.maximum(np.nan_to_num(_n(frame,'asof_pitcher_n',0),nan=0.0),0);bn=np.maximum(np.nan_to_num(_n(frame,'asof_batter_n',0),nan=0.0),0)\n    career=_n(frame,'asof_pitcher_success_rate',.5);career=np.where(np.isfinite(career),career,.5)\n    p1=_n(frame,'asof_pitcher_prev1_game_success_rate');p3=_n(frame,'asof_pitcher_prev3_game_success_rate');p5=_n(frame,'asof_pitcher_prev5_game_success_rate')\n    p1=np.where(np.isfinite(p1),p1,career);p3=np.where(np.isfinite(p3),p3,p1);p5=np.where(np.isfinite(p5),p5,p3)\n    bsucc=_n(frame,'asof_batter_success_rate',.5);bsucc=np.where(np.isfinite(bsucc),bsucc,.5)\n    middle=_n(frame,'asof_pitcher_middle_rate',0);reverse=_n(frame,'asof_pitcher_reverse_rate',0);ball=_n(frame,'asof_pitcher_ball_rate',0);strike=_n(frame,'asof_pitcher_strike_rate',0)\n    fast=_n(frame,'asof_pitcher_fastball_rate',1/3);brk=_n(frame,'asof_pitcher_breaking_rate',1/3);off=_n(frame,'asof_pitcher_offspeed_rate',1/3)\n    fast=np.where(np.isfinite(fast),fast,1/3);brk=np.where(np.isfinite(brk),brk,1/3);off=np.where(np.isfinite(off),off,1/3);sm=np.maximum(fast+brk+off,1e-6);fast/=sm;brk/=sm;off/=sm\n    balls=np.nan_to_num(_n(frame,'balls_before',0));strikes=np.nan_to_num(_n(frame,'strikes_before',0));inning=np.nan_to_num(_n(frame,'inning',1));score=np.nan_to_num(_n(frame,'score_diff_pitcher_team',0));li=np.maximum(np.nan_to_num(_n(frame,'li',0)),0)\n    top=frame['top_bottom'].astype(str).to_numpy();hwe=np.nan_to_num(_n(frame,'home_win_expectancy',50),nan=50)/100.;awe=np.nan_to_num(_n(frame,'away_win_expectancy',50),nan=50)/100.;pwe=np.where(top=='T',hwe,awe)\n    ph=frame['pitcher_hand'].astype(str).to_numpy();bh=frame['batter_hand'].astype(str).to_numpy()\n    z['self__log_pitcher_n']=np.log1p(pn).astype(np.float32);z['self__pitcher_reliability']=(pn/(pn+120)).astype(np.float32)\n    z['self__career_success']=career.astype(np.float32);z['self__prev1_success']=p1.astype(np.float32);z['self__prev3_success']=p3.astype(np.float32);z['self__prev5_success']=p5.astype(np.float32)\n    z['self__recent_blend']=(.50*p1+.30*p3+.20*p5).astype(np.float32);z['self__prev1_minus_prev5']=(p1-p5).astype(np.float32);z['self__recent_minus_career']=(.5*p1+.3*p3+.2*p5-career).astype(np.float32)\n    middle_safe=np.nan_to_num(middle,nan=0.0);m1=_n(frame,'asof_pitcher_prev1_game_middle_rate');m3=_n(frame,'asof_pitcher_prev3_game_middle_rate');m5=_n(frame,'asof_pitcher_prev5_game_middle_rate');m1=np.where(np.isfinite(m1),m1,middle_safe);m3=np.where(np.isfinite(m3),m3,m1);m5=np.where(np.isfinite(m5),m5,m3);z['self__recent_middle_blend']=(.50*m1+.30*m3+.20*m5).astype(np.float32);z['self__recent_middle_delta']=(m1-m5).astype(np.float32)\n    z['self__log_batter_n']=np.log1p(bn).astype(np.float32);z['self__batter_reliability']=(bn/(bn+120)).astype(np.float32);z['self__batter_success']=bsucc.astype(np.float32);z['self__pitcher_minus_batter']=(career-bsucc).astype(np.float32)\n    z['self__middle']=np.nan_to_num(middle,nan=0).astype(np.float32);z['self__reverse']=np.nan_to_num(reverse,nan=0).astype(np.float32);z['self__strike_ball_margin']=np.nan_to_num(strike-ball,nan=0).astype(np.float32)\n    entropy=-(fast*np.log(np.clip(fast,1e-6,1))+brk*np.log(np.clip(brk,1e-6,1))+off*np.log(np.clip(off,1e-6,1)))\n    z['self__pitchmix_entropy']=entropy.astype(np.float32);z['self__offspeed']=off.astype(np.float32);z['self__fast_off_logratio']=np.log(np.clip(fast,1e-6,1)/np.clip(off,1e-6,1)).astype(np.float32);z['self__break_off_logratio']=np.log(np.clip(brk,1e-6,1)/np.clip(off,1e-6,1)).astype(np.float32)\n    z['self__count_pressure']=(.7*(balls==3)+1.1*((balls==3)&(strikes==2))+.25*(strikes==2)).astype(np.float32);z['self__full_count']=((balls==3)&(strikes==2)).astype(np.float32)\n    z['self__inning']=inning.astype(np.float32);z['self__late']=(inning>=7).astype(np.float32);z['self__score_abs']=np.abs(score).astype(np.float32);z['self__li_log1p']=np.log1p(li).astype(np.float32)\n    z['self__pitcher_we']=np.clip(pwe,0,1).astype(np.float32);z['self__win_uncertainty']=(1-2*np.abs(np.clip(pwe,0,1)-.5)).astype(np.float32);z['self__same_hand']=(ph==bh).astype(np.float32)\n    pressure=z['self__count_pressure'].to_numpy(np.float64);z['self__career_x_pressure']=(career*pressure).astype(np.float32);z['self__recent_x_pressure']=((.5*p1+.3*p3+.2*p5)*pressure).astype(np.float32);z['self__offspeed_x_same_hand']=(off*(ph==bh)).astype(np.float32)\n    return z\n\n# ------------------------------------------------------------------\n# Global-ASOF-aligned current-season decomposition.\n# Official asof_n is global across R/F, so historical subtraction MUST\n# use all official history, not one routed domain.\n# ------------------------------------------------------------------\nP_COMPONENT_RATES=['asof_pitcher_reverse_rate','asof_pitcher_middle_rate','asof_pitcher_ball_rate','asof_pitcher_strike_rate','asof_pitcher_fastball_rate','asof_pitcher_breaking_rate','asof_pitcher_offspeed_rate']\nB_COMPONENT_RATES=['asof_batter_middle_rate']\n\ndef _rounded_cumulative_count(n,rate):\n    n=np.asarray(n,np.float64);rate=np.asarray(rate,np.float64);out=np.zeros(len(n),np.float64);ok=(n>0)&np.isfinite(rate);out[ok]=np.rint(n[ok]*rate[ok]);return out\n\ndef build_global_season_bank(history,cutoff):\n    h=history.loc[history.season.astype(int)<int(cutoff)].copy()\n    prior=float(h.loc[h.season.eq(h.season.max()),TARGET].mean()) if len(h) else .5\n    pg=h.groupby('pitcher_id',sort=False)[TARGET].agg(['size','sum']) if len(h) else pd.DataFrame(columns=['size','sum'])\n    bg=h.groupby('batter_id',sort=False)[TARGET].agg(['size','sum']) if len(h) else pd.DataFrame(columns=['size','sum'])\n    bank={'cutoff':int(cutoff),'prior':prior,'pitcher_n':pg['size'].to_dict(),'pitcher_s':pg['sum'].to_dict(),'batter_n':bg['size'].to_dict(),'batter_s':bg['sum'].to_dict(),'source_max_season':int(h.season.max()) if len(h) else None}\n\n    # Component-rate lineage: asof_n * rate is a rounded cumulative count. The last\n    # historical row stores counts BEFORE that last pitch. We therefore use that\n    # pre-last boundary as the baseline. The resulting window contains exactly one\n    # boundary pitch + current-season pitches, avoiding any need to inspect other test rows.\n    prev=h.loc[h.season.eq(h.season.max())] if len(h) else h\n    bank['component_priors']={c:float(pd.to_numeric(prev[c],errors='coerce').mean()) if c in prev and pd.to_numeric(prev[c],errors='coerce').notna().any() else 0.5 for c in P_COMPONENT_RATES+B_COMPONENT_RATES}\n    if len(h):\n        pidx=h.groupby('pitcher_id',sort=False)['asof_pitcher_n'].idxmax();plast=h.loc[pidx,['pitcher_id','asof_pitcher_n']+P_COMPONENT_RATES].set_index('pitcher_id')\n        pbase_n=pd.to_numeric(plast['asof_pitcher_n'],errors='coerce').fillna(0).astype(np.float64);bank['pitcher_component_base_n']=pbase_n.to_dict();bank['pitcher_component_base']={}\n        for c in P_COMPONENT_RATES:\n            rr=pd.to_numeric(plast[c],errors='coerce');cnt=np.rint(pbase_n*rr);bank['pitcher_component_base'][c]=cnt.to_dict()\n        bidx=h.groupby('batter_id',sort=False)['asof_batter_n'].idxmax();blast=h.loc[bidx,['batter_id','asof_batter_n']+B_COMPONENT_RATES].set_index('batter_id')\n        bbase_n=pd.to_numeric(blast['asof_batter_n'],errors='coerce').fillna(0).astype(np.float64);bank['batter_component_base_n']=bbase_n.to_dict();bank['batter_component_base']={}\n        for c in B_COMPONENT_RATES:\n            rr=pd.to_numeric(blast[c],errors='coerce');cnt=np.rint(bbase_n*rr);bank['batter_component_base'][c]=cnt.to_dict()\n    else:\n        bank['pitcher_component_base_n']={};bank['pitcher_component_base']={c:{} for c in P_COMPONENT_RATES};bank['batter_component_base_n']={};bank['batter_component_base']={c:{} for c in B_COMPONENT_RATES}\n    return bank\n\ndef transform_global_season(bank,rows):\n    r=rows.reset_index(drop=True);z=pd.DataFrame(index=np.arange(len(r)));prior=float(bank['prior'])\n    def one(ent,ncol,rcol,prefix,kn=80.0):\n        ids=r[ent];hn=ids.map(bank[prefix+'_n']).fillna(0).to_numpy(np.float64);hs=ids.map(bank[prefix+'_s']).fillna(0).to_numpy(np.float64)\n        an=np.maximum(np.nan_to_num(_n(r,ncol,0),nan=0),0);ar=_n(r,rcol,prior);ar=np.where(np.isfinite(ar),ar,prior)\n        # Exact integer reconstruction: official rates are rounded, but n*rate is always\n        # within <0.01 of the underlying integer count in the released train.\n        cumulative_success=_rounded_cumulative_count(an,ar)\n        sn=np.maximum(an-hn,0.0);ss=np.clip(cumulative_success-hs,0.0,sn)\n        raw=np.divide(ss,sn,out=np.full(len(r),prior,np.float64),where=sn>0)\n        for k in [40.,80.,160.]:z[f'gseason__{prefix}_rate_k{int(k)}']=((ss+k*prior)/(sn+k)).astype(np.float32)\n        z[f'gseason__{prefix}_raw']=raw.astype(np.float32);z[f'gseason__{prefix}_log_n']=np.log1p(sn).astype(np.float32);z[f'gseason__{prefix}_reliability']=(sn/(sn+kn)).astype(np.float32);z[f'gseason__{prefix}_minus_career']=(raw-ar).astype(np.float32)\n    one('pitcher_id','asof_pitcher_n','asof_pitcher_success_rate','pitcher',80.)\n    one('batter_id','asof_batter_n','asof_batter_success_rate','batter',100.)\n    z['gseason__pitcher_minus_batter_k80']=(z['gseason__pitcher_rate_k80']-z['gseason__batter_rate_k80']).astype(np.float32)\n\n    # Exact cumulative component counts with one-pitch boundary bridge.\n    ids=r['pitcher_id'];an=np.maximum(np.nan_to_num(_n(r,'asof_pitcher_n',0),nan=0),0);base_n=ids.map(bank['pitcher_component_base_n']).fillna(0).to_numpy(np.float64);win_n=np.maximum(an-base_n,0.0)\n    comp={}\n    for c in P_COMPONENT_RATES:\n        pr=float(bank['component_priors'].get(c,0.5));rr=_n(r,c,pr);rr=np.where(np.isfinite(rr),rr,pr);cum=_rounded_cumulative_count(an,rr);base=ids.map(bank['pitcher_component_base'].get(c,{})).fillna(0).to_numpy(np.float64);wc=np.clip(cum-base,0.0,win_n);comp[c]=((wc+60.0*pr)/(win_n+60.0)).astype(np.float32)\n    z['gseason__pitcher_component_log_n']=np.log1p(win_n).astype(np.float32);z['gseason__pitcher_component_reliability']=(win_n/(win_n+60.)).astype(np.float32)\n    z['gseason__pitcher_strike_rate']=comp['asof_pitcher_strike_rate'];z['gseason__pitcher_ball_rate']=comp['asof_pitcher_ball_rate'];z['gseason__pitcher_strike_ball_margin']=(comp['asof_pitcher_strike_rate']-comp['asof_pitcher_ball_rate']).astype(np.float32)\n    z['gseason__pitcher_middle_rate']=comp['asof_pitcher_middle_rate'];z['gseason__pitcher_reverse_rate']=comp['asof_pitcher_reverse_rate']\n    fast=np.clip(comp['asof_pitcher_fastball_rate'],1e-6,1);brk=np.clip(comp['asof_pitcher_breaking_rate'],1e-6,1);off=np.clip(comp['asof_pitcher_offspeed_rate'],1e-6,1);sm=np.maximum(fast+brk+off,1e-6);fast=fast/sm;brk=brk/sm;off=off/sm\n    z['gseason__pitcher_fastball_rate']=fast.astype(np.float32);z['gseason__pitcher_breaking_rate']=brk.astype(np.float32);z['gseason__pitcher_offspeed_rate']=off.astype(np.float32);z['gseason__pitcher_pitchmix_entropy']=(-(fast*np.log(fast)+brk*np.log(brk)+off*np.log(off))).astype(np.float32)\n\n    bids=r['batter_id'];ban=np.maximum(np.nan_to_num(_n(r,'asof_batter_n',0),nan=0),0);bbase_n=bids.map(bank['batter_component_base_n']).fillna(0).to_numpy(np.float64);bwin=np.maximum(ban-bbase_n,0.0);c='asof_batter_middle_rate';pr=float(bank['component_priors'].get(c,0.5));rr=_n(r,c,pr);rr=np.where(np.isfinite(rr),rr,pr);cum=_rounded_cumulative_count(ban,rr);base=bids.map(bank['batter_component_base'].get(c,{})).fillna(0).to_numpy(np.float64);wc=np.clip(cum-base,0.0,bwin);bmid=((wc+80.*pr)/(bwin+80.)).astype(np.float32)\n    z['gseason__batter_middle_rate']=bmid;z['gseason__batter_middle_minus_pitcher_middle']=(bmid-z['gseason__pitcher_middle_rate']).astype(np.float32)\n    return z\n\ndef global_season_features_for_training(frame):\n    pieces=[]\n    for year in sorted(frame.season.astype(int).unique()):\n        idx=frame.index[frame.season.astype(int).eq(year)];bank=build_global_season_bank(frame.loc[frame.season.astype(int)<year],year);part=transform_global_season(bank,frame.loc[idx]);part.index=idx;pieces.append(part)\n    return pd.concat(pieces).sort_index().reindex(frame.index).reset_index(drop=True)\n\n\nprint('Exact-ASOF row-state functions loaded')\n\n\ndef add_eng(frame):\n    x=frame.copy()\n    x['eng_recent35']=(.5*pd.to_numeric(x.asof_pitcher_prev1_game_success_rate,errors='coerce')+.3*pd.to_numeric(x.asof_pitcher_prev3_game_success_rate,errors='coerce')+.2*pd.to_numeric(x.asof_pitcher_prev5_game_success_rate,errors='coerce'))\n    x['eng_recent_delta']=x.eng_recent35-pd.to_numeric(x.asof_pitcher_success_rate,errors='coerce')\n    x['eng_sb']=pd.to_numeric(x.asof_pitcher_strike_rate,errors='coerce')-pd.to_numeric(x.asof_pitcher_ball_rate,errors='coerce')\n    x['eng_same']=(x.pitcher_hand.astype(str)==x.batter_hand.astype(str)).astype(np.float32)\n    x['eng_close']=(pd.to_numeric(x.score_diff_pitcher_team,errors='coerce').abs()<=1).astype(np.float32)\n    x['eng_late']=(pd.to_numeric(x.inning,errors='coerce')>=7).astype(np.float32)\n    return x\n\ndef encode_with_maps(frame,maps):\n    out=pd.DataFrame(index=np.arange(len(frame)))\n    for c,mp in maps.items():\n        out['code_'+c]=frame[c].astype('string').fillna('<NA>').astype(str).map(mp).fillna(-1).astype(np.float32)\n    return out\n\ndef transform_base(test):\n    rawnum=META['rawnum']\n    selfx=build_self_state(test)\n    gx=transform_global_season(META['target_global_bank'],test)\n    B0=pd.concat([test[rawnum].apply(pd.to_numeric,errors='coerce').reset_index(drop=True),selfx.reset_index(drop=True),gx.reset_index(drop=True)],axis=1)\n\n    cb=encode_with_maps(test,META['exact_cat_maps'])\n    B_exact=pd.concat([B0,cb],axis=1)\n    B_exact=B_exact.reindex(columns=META['exact_feature_cols'])\n    m1=joblib.load(MODEL/'base_exact_lgb.joblib')\n    p1=m1.predict_proba(B_exact)[:,1]\n\n    B=B0.reindex(columns=META['ridge_feature_cols']).replace([np.inf,-np.inf],np.nan)\n    med=pd.Series(META['ridge_med'])\n    B=B.fillna(med).to_numpy(np.float32)\n    mu=np.asarray(META['ridge_mu'],dtype=np.float32);sd=np.asarray(META['ridge_sd'],dtype=np.float32)\n    B=np.clip((B-mu)/sd,-8,8)\n    m2=joblib.load(MODEL/'base_ridge.joblib')\n    p2=np.clip(m2.predict(B),.001,.999)\n\n    t=add_eng(test)\n    Bnum=t[META['trend_num_cols']].apply(pd.to_numeric,errors='coerce').reset_index(drop=True)\n    cb=encode_with_maps(t,META['trend_cat_maps'])\n    Bt=pd.concat([Bnum,cb],axis=1).astype(np.float32).reindex(columns=META['trend_feature_cols'])\n    m3=joblib.load(MODEL/'base_trend_lgb.joblib')\n    prior=META['trend_prior']\n    bt=np.array([float(prior[d]) for d in test.domain3.to_numpy()],dtype=np.float64)\n    p3=np.clip(bt+m3.predict(Bt),.001,.999)\n    return np.clip((np.asarray(p1,float)+np.asarray(p2,float)+np.asarray(p3,float))/3.0,0,1)\n\ndef transform_adapter(rows,base_pred):\n    prep=META['adapter_prep'];r=add_eng(rows.reset_index(drop=True));A=r[prep['cols']].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)\n    for c in prep['cols']:\n        A[c]=A[c].fillna(float(prep['med'].get(c,0.))).astype(np.float32)\n    for c,mp in prep['maps'].items():\n        A['code_'+c]=r[c].astype('string').fillna('<NA>').astype(str).map(mp).fillna(-1).astype(np.float32)\n    A['base_pred']=np.asarray(base_pred,dtype=np.float32)\n    A=A.reindex(columns=META['adapter_base_feature_cols'])\n    if not np.isfinite(A.to_numpy(float)).all():\n        raise RuntimeError('non-finite adapter input')\n    return A\n\n\n# ---------------- V15 selected challenger blocks ----------------\ndef _ctx_frame_v15(frame):\n    x=frame.copy()\n    x['v15_count']=x['balls_before'].astype(str)+'_'+x['strikes_before'].astype(str)\n    x['v15_inning_phase']=pd.cut(pd.to_numeric(x['inning'],errors='coerce'),[-np.inf,3,6,9,np.inf],labels=['early','middle','late','extra']).astype(str)\n    x['v15_pressure']=np.where((x['balls_before']==3)&(x['strikes_before']==2),'full',np.where(x['balls_before']==3,'three_ball',np.where(x['strikes_before']==2,'two_strike','normal')))\n    x['v15_runners']=np.where(pd.to_numeric(x['num_runners_on'],errors='coerce').fillna(0)>0,'on','empty')\n    x['v15_li_bucket']=pd.cut(pd.to_numeric(x['li'],errors='coerce'),[-np.inf,.7,1.5,np.inf],labels=['low','mid','high']).astype(str)\n    return x\n\ndef _lookup_v15(table,rows,keys,field,default=0.0):\n    if len(keys)==1:\n        s=table.set_index(keys[0])[field]\n        return rows[keys[0]].map(s).fillna(default).to_numpy(np.float32)\n    idx=pd.MultiIndex.from_frame(rows[keys])\n    s=table.set_index(keys)[field]\n    return s.reindex(idx).fillna(default).to_numpy(np.float32)\n\ndef transform_pitcher_conditional(rows):\n    r=_ctx_frame_v15(rows.reset_index(drop=True));parts=[]\n    for spec in META['pitcher_conditional_specs']:\n        t=spec['table'];ctx=spec['ctx'];prefix=spec['prefix'];z=pd.DataFrame(index=np.arange(len(r)))\n        keys=['pitcher_id',ctx]\n        for f in ('rate','reliability','delta'):\n            z[prefix+'__'+f]=_lookup_v15(t,r,keys,f,0.0)\n        parts.append(z)\n    out=pd.concat(parts,axis=1).astype(np.float32)\n    return out.reindex(columns=META['pitcher_conditional_cols'])\n\ndef transform_exact_adapter_block(rows):\n    sx=build_self_state(rows).add_prefix('exact__')\n    gx=transform_global_season(META['target_global_bank'],rows).add_prefix('exact__')\n    out=pd.concat([sx.reset_index(drop=True),gx.reset_index(drop=True)],axis=1)\n    out=out.replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)\n    return out.reindex(columns=META['exact_adapter_cols'])\n\ndef transform_v15_adapter(rows,base_pred):\n    base=transform_adapter(rows,base_pred)\n    pc=transform_pitcher_conditional(rows)\n    ex=transform_exact_adapter_block(rows)\n    X=pd.concat([base.reset_index(drop=True),pc.reset_index(drop=True),ex.reset_index(drop=True)],axis=1)\n    X=X.reindex(columns=META['adapter_feature_cols'])\n    if X.isna().any().any() or not np.isfinite(X.to_numpy(float)).all():\n        raise RuntimeError('non-finite V15 adapter input')\n    return X\n\n\n# ---------------- V18 strict-temporal stable conditional correction ----------------\ndef _v18_pressure_frame(rows):\n    r=rows.copy()\n    balls=pd.to_numeric(r['balls_before'],errors='coerce').fillna(0).astype(int)\n    strikes=pd.to_numeric(r['strikes_before'],errors='coerce').fillna(0).astype(int)\n    r['v18_pressure']=np.where((balls==3)&(strikes==2),'full',np.where(balls==3,'3ball',np.where(strikes==2,'2strike','normal')))\n    r['v18_pressure_bhand']=r['v18_pressure'].astype(str)+'|'+r['batter_hand'].astype(str)\n    return r\n\ndef transform_v18_stable(rows):\n    r=_v18_pressure_frame(rows.reset_index(drop=True));parts=[]\n    for spec in META['v18_stable_specs']:\n        t=spec['table'];ctx=spec['ctx'];prefix=spec['prefix'];z=pd.DataFrame(index=np.arange(len(r)))\n        de=_lookup_v15(t,r,['pitcher_id',ctx],'delta',0.0).astype(np.float32)\n        re=_lookup_v15(t,r,['pitcher_id',ctx],'reliability',0.0).astype(np.float32)\n        z[prefix+'__delta']=de\n        z[prefix+'__weighted_delta']=(de*re).astype(np.float32)\n        z[prefix+'__reliability']=re\n        parts.append(z)\n    z=pd.concat(parts,axis=1)\n    balls=pd.to_numeric(r['balls_before'],errors='coerce').fillna(0).astype(int).to_numpy()\n    strikes=pd.to_numeric(r['strikes_before'],errors='coerce').fillna(0).astype(int).to_numpy()\n    for b in range(4): z[f'ctx__ball_{b}']=(balls==b).astype(np.float32)\n    for s in range(3): z[f'ctx__strike_{s}']=(strikes==s).astype(np.float32)\n    for b in range(4):\n        for s in range(3): z[f'ctx__count_{b}_{s}']=((balls==b)&(strikes==s)).astype(np.float32)\n    z=z.reindex(columns=META['v18_stable_cols']).replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)\n    if not np.isfinite(z.to_numpy(float)).all(): raise RuntimeError('non-finite V18 stable features')\n    return z\n\ndef main():\n    test=pd.read_csv(DATA/'test.csv',encoding='utf-8-sig',low_memory=False)\n    if TARGET in test.columns: raise RuntimeError('test.csv unexpectedly contains target')\n    required=['row_id','season','pitcher_id','batter_id','game_type','pitcher_team_id','batter_team_id']\n    missing=[c for c in required if c not in test.columns]\n    if missing: raise RuntimeError(f'missing test columns: {missing}')\n    test['season']=pd.to_numeric(test['season'],errors='raise').astype(int)\n    test['game_type']=test['game_type'].astype('string').fillna('<NA>').astype(str)\n    expected=int(META['expected_year'])\n    if bool((test.season!=expected).any()): raise RuntimeError(f'evaluation season differs from expected {expected}')\n    if bool((~test.game_type.isin(['R','F'])).any()): raise RuntimeError('unexpected game_type')\n    anchor=int(META['anchor_team_id'])\n    test['domain3']=np.where(test.game_type.eq('F'),'F',np.where((test.pitcher_team_id==anchor)|(test.batter_team_id==anchor),'R_ANCHOR','R_CORE'))\n\n    base=transform_base(test)\n    X=transform_v15_adapter(test,base)\n    corr=np.zeros(len(test),dtype=np.float64)\n    seeds=META['adapter_seeds']\n    for seed in seeds:\n        c=np.zeros(len(test),dtype=np.float64)\n        for d in ['R_CORE','R_ANCHOR','F']:\n            mask=test.domain3.eq(d).to_numpy()\n            if not mask.any(): continue\n            m=joblib.load(MODEL/f'adapter_seed{int(seed)}_{d}.joblib')\n            c[mask]=m.predict(X.loc[mask])\n        corr+=c/len(seeds)\n    final=np.clip(base+float(META['adapter_shrink'])*corr,0,1)\n    if META.get('v18_stable_enabled',False):\n        Xs=transform_v18_stable(test)\n        sm=joblib.load(MODEL/META['v18_stable_model_file'])\n        arr=Xs.to_numpy(np.float32);mu=np.asarray(META['v18_stable_mu'],dtype=np.float32);sd=np.asarray(META['v18_stable_sd'],dtype=np.float32);arr=np.clip((arr-mu)/sd,-8,8)\n        sc=np.asarray(sm.predict(arr),dtype=np.float64)\n        core=test.domain3.eq(META.get('v18_route_domain','R_CORE')).to_numpy()\n        final[core]=np.clip(final[core]+float(META['v18_stable_gamma'])*sc[core],0,1)\n    if len(final)!=len(test) or not np.isfinite(final).all() or (final<0).any() or (final>1).any():\n        raise RuntimeError('invalid prediction vector')\n    out=pd.DataFrame({'row_id':test.row_id.to_numpy(),'control_success':final})\n    out.to_csv(OUT/'submission.csv',index=False)\n    print('V18_TEMPORAL_STABLE_INFERENCE_OK',len(out),float(final.min()),float(final.max()),float(final.mean()))\n\nif __name__=='__main__':\n    main()\n"
compile(SCRIPT_SOURCE,'script.py','exec')
(WORK_ROOT/'script.py').write_text(SCRIPT_SOURCE,encoding='utf-8')
req=f"lightgbm=={lgb.__version__}\nscikit-learn=={sklearn.__version__}\njoblib=={joblib.__version__}\n"
(WORK_ROOT/'requirements.txt').write_text(req,encoding='utf-8')
print('requirements:\n'+req)

# ---------- 5) working-tree smoke: packaged script must match notebook prediction ----------
smoke=WORK_ROOT/'_smoke';(smoke/'data').mkdir(parents=True,exist_ok=True)
shutil.copytree(MODEL_DIR,smoke/'model');shutil.copy2(WORK_ROOT/'script.py',smoke/'script.py');shutil.copy2(TEST_PATH,smoke/'data'/'test.csv')
r=subprocess.run([sys.executable,'script.py'],cwd=smoke,text=True,capture_output=True,check=False);print(r.stdout)
if r.returncode!=0:raise RuntimeError('working-tree smoke failed:\n'+r.stderr)
smoke_out=pd.read_csv(smoke/'output'/'submission.csv')
assert list(smoke_out.columns)==['row_id','control_success'] and len(smoke_out)==len(test)
assert np.array_equal(smoke_out.row_id.astype(str).to_numpy(),test.row_id.astype(str).to_numpy())
maxdiff=float(np.max(np.abs(smoke_out.control_success.to_numpy(float)-final_direct))) if len(test) else 0.0
print('WORKING_TREE_SMOKE_OK max_abs_diff',maxdiff);assert maxdiff<1e-10

# ---------- 6) exact ZIP root + CRC + extracted inference smoke ----------
zip_path=WORK_ROOT/'submit.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    z.write(WORK_ROOT/'script.py','script.py');z.write(WORK_ROOT/'requirements.txt','requirements.txt')
    for p in sorted(MODEL_DIR.rglob('*')):
        if p.is_file():z.write(p,'model/'+str(p.relative_to(MODEL_DIR)))
with zipfile.ZipFile(zip_path) as z:
    if z.testzip() is not None:raise RuntimeError('ZIP CRC failure')
    roots=sorted({n.split('/')[0] for n in z.namelist()});assert roots==['model','requirements.txt','script.py'],roots
print('ZIP_STRUCTURE_AND_CRC_OK',round(zip_path.stat().st_size/1e6,2),'MB')
extract=WORK_ROOT/'_extract'
if extract.exists():shutil.rmtree(extract)
with zipfile.ZipFile(zip_path) as z:z.extractall(extract)
(extract/'data').mkdir(exist_ok=True);shutil.copy2(TEST_PATH,extract/'data'/'test.csv')
r2=subprocess.run([sys.executable,'script.py'],cwd=extract,text=True,capture_output=True,check=False);print(r2.stdout)
if r2.returncode!=0:raise RuntimeError('extracted ZIP smoke failed:\n'+r2.stderr)
out2=pd.read_csv(extract/'output'/'submission.csv')
assert len(out2)==len(test) and out2.control_success.between(0,1).all() and np.isfinite(out2.control_success).all()
assert np.max(np.abs(out2.control_success.to_numpy(float)-final_direct))<1e-10
print('EXTRACTED_ZIP_SMOKE_OK')

try:
    top=Path('/kaggle/working/v18_interim_submit.zip')
    if top.resolve()!=zip_path.resolve():shutil.copy2(zip_path,top)
    print('TOP_LEVEL_SUBMIT_COPY:',top)
except Exception as e:print('top-level copy skipped:',repr(e))

print('V18_BASE_PACKAGE_READY_FOR_V20')
print('V18_BASE_SUBMISSION_TESTS_PASSED')
print('V18 INTERIM ZIP:',zip_path)
